<a href="https://colab.research.google.com/github/kvanoost/AGP/blob/main/2_create_predictor_cache_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kasai modelling cache — annual Google Satellite Embeddings at observation year

This notebook reads `NN/Data/Kasai_reference.csv`, extracts the annual Google Satellite Embedding vector for each observation point at its observation year, and saves the cache under:

`NN/Data/Cache/<MODEL_NAME>/`

No model fitting is done here. This notebook only creates the first reusable cache.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json
from pathlib import Path

# Change this to your notebook path in Google Drive
nb_path = Path("/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN.ipynb")

nb = json.loads(nb_path.read_text(encoding="utf-8"))

# Remove broken widget metadata
nb.get("metadata", {}).pop("widgets", None)

fixed_path = nb_path.with_name(nb_path.stem + "_fixed.ipynb")
fixed_path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8")

print("Fixed notebook saved as:")
print(fixed_path)

# 1. Setup

## 1.1 Install and import packages

In [ ]:
# ------------------------------------------------------------
# Install/import packages
# ------------------------------------------------------------
import os
import time
import pandas as pd
import numpy as np

from google.colab import drive

try:
    import geemap
except ImportError:
    !pip -q install geemap
    import geemap

print('Packages imported.')

Packages imported.


## 1.2 Mount Google Drive

In [ ]:
# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.3 Earth Engine authentication and initialization

In [ ]:
# ------------------------------------------------------------
# Earth Engine auth/init
# ------------------------------------------------------------
import ee
try:
    ee.Initialize(project='panafricanlu')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='panafricanlu')
print('Earth Engine initialized.')

Earth Engine initialized.


## 1.4 Project paths and cache folder

In [ ]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN'
DATA_DIR = os.path.join(PROJECT_ROOT, 'Data')

# Change this model/cache name later if you want a different cache family.
MODEL_NAME = 'annual_satellite_embeddings_obs_year'

CACHE_DIR = os.path.join(DATA_DIR, 'Cache', MODEL_NAME)
os.makedirs(CACHE_DIR, exist_ok=True)

REFERENCE_CSV = os.path.join(DATA_DIR, 'Kasai_reference.csv')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('CACHE_DIR:', CACHE_DIR)
print('REFERENCE_CSV:', REFERENCE_CSV)

PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv


## 2.1 Load `Kasai_reference.csv`

In [ ]:
# ------------------------------------------------------------
# Load reference CSV
# ------------------------------------------------------------
if not os.path.exists(REFERENCE_CSV):
    raise FileNotFoundError(f'Could not find reference CSV: {REFERENCE_CSV}')

ref_df = pd.read_csv(REFERENCE_CSV)

print('Reference table shape:', ref_df.shape)
print('Columns:')
print(list(ref_df.columns))

display(ref_df.head())

Reference table shape: (3706, 9)
Columns:
['unit_id', 'latitude', 'longitude', 'observation_year', 'observation_month', 'zone_id', 'land_cover', 'source_index', 'observation_yearmonth_raw']


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04


## 2.2 Check required columns and create `sample_id`

In [ ]:
# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------
required_cols = [
    'unit_id',
    'latitude',
    'longitude',
    'observation_year'
]

missing_cols = [c for c in required_cols if c not in ref_df.columns]

if missing_cols:
    raise ValueError(
        'Missing required columns in Kasai_reference.csv: '
        + ', '.join(missing_cols)
    )

ref_df = ref_df.copy()
ref_df['unit_id'] = ref_df['unit_id'].astype(str)
ref_df['observation_year'] = ref_df['observation_year'].astype(int)
ref_df['latitude'] = pd.to_numeric(ref_df['latitude'], errors='coerce')
ref_df['longitude'] = pd.to_numeric(ref_df['longitude'], errors='coerce')

if 'observation_month' in ref_df.columns:
    ref_df['observation_month'] = ref_df['observation_month'].astype(str)
else:
    ref_df['observation_month'] = ''

if 'land_cover' not in ref_df.columns:
    ref_df['land_cover'] = ''

if 'zone_id' not in ref_df.columns:
    ref_df['zone_id'] = 'Kasai'

# Drop records without usable coordinates.
coord_missing = ref_df['latitude'].isna() | ref_df['longitude'].isna()
if coord_missing.any():
    print('Dropping rows with missing coordinates:', int(coord_missing.sum()))
    display(ref_df.loc[coord_missing].head())
    ref_df = ref_df.loc[~coord_missing].copy()

# Unique sample ID for this observation-year cache.
ref_df['sample_id'] = (
    ref_df['unit_id'].astype(str)
    + '_'
    + ref_df['observation_year'].astype(str)
)

print('Rows:', len(ref_df))
print('Unique unit_id:', ref_df['unit_id'].nunique())
print('Unique sample_id:', ref_df['sample_id'].nunique())
print('Observation years:')
display(ref_df['observation_year'].value_counts().sort_index().reset_index(name='n'))

display(ref_df.head())

Rows: 3706
Unique unit_id: 3706
Unique sample_id: 3706
Observation years:


,observation_year,n
0,2020,1180
1,2021,675
2,2022,733
3,2023,362
4,2024,432
5,2025,324


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020


## 3.1 Embedding settings and band names

In [ ]:
# ------------------------------------------------------------
# Embedding settings
# ------------------------------------------------------------
EMBEDDING_COLLECTION = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'

# Google Satellite Embeddings are available annually from 2017 onward.
EMBEDDING_START_YEAR = 2017
EMBEDDING_END_YEAR = 2025

SCALE = 10
TILE_SCALE = 4

sat_embedding_annual = ee.ImageCollection(EMBEDDING_COLLECTION)

# Inspect band names from the first available annual image.
first_embedding_img = (
    sat_embedding_annual
    .filterDate(f'{EMBEDDING_START_YEAR}-01-01', f'{EMBEDDING_START_YEAR + 1}-01-01')
    .first()
)

embedding_bands = first_embedding_img.bandNames().getInfo()

print('Number of embedding bands:', len(embedding_bands))
print('First bands:', embedding_bands[:10])
print('Last bands:', embedding_bands[-5:])

Number of embedding bands: 64
First bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']
Last bands: ['A59', 'A60', 'A61', 'A62', 'A63']


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


## 3.2 Keep points with observation years covered by embeddings

In [ ]:
# ------------------------------------------------------------
# Filter to available embedding years
# ------------------------------------------------------------
cache_input_df = ref_df[
    (ref_df['observation_year'] >= EMBEDDING_START_YEAR)
    & (ref_df['observation_year'] <= EMBEDDING_END_YEAR)
].copy()

excluded_df = ref_df[
    ~(
        (ref_df['observation_year'] >= EMBEDDING_START_YEAR)
        & (ref_df['observation_year'] <= EMBEDDING_END_YEAR)
    )
].copy()

print('Rows kept for embedding extraction:', len(cache_input_df))
print('Rows excluded because observation year is outside embedding period:', len(excluded_df))

if len(excluded_df) > 0:
    display(excluded_df[['unit_id', 'observation_year', 'latitude', 'longitude']].head())

if cache_input_df.empty:
    raise ValueError('No points fall within the available annual embedding period.')

Rows kept for embedding extraction: 3706
Rows excluded because observation year is outside embedding period: 0


## 3.3 Helper functions for Earth Engine extraction

In [ ]:
# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def dataframe_to_ee_points(df):
    features = []

    for _, row in df.iterrows():
        props = {
            'sample_id': str(row['sample_id']),
            'unit_id': str(row['unit_id']),
            'zone_id': str(row.get('zone_id', '')),
            'land_cover': str(row.get('land_cover', '')),
            'observation_year': int(row['observation_year']),
            'observation_month': str(row.get('observation_month', '')),
            'latitude': float(row['latitude']),
            'longitude': float(row['longitude'])
        }

        geom = ee.Geometry.Point([
            float(row['longitude']),
            float(row['latitude'])
        ])

        features.append(ee.Feature(geom, props))

    return ee.FeatureCollection(features)


def get_annual_embedding_image(year):
    start = ee.Date.fromYMD(int(year), 1, 1)
    end = start.advance(1, 'year')

    img = (
        sat_embedding_annual
        .filterDate(start, end)
        .mosaic()
        .select(embedding_bands)
    )

    return img


def extract_embeddings_for_year(year, year_df):
    print(f'Extracting embeddings for {year} | points: {len(year_df)}')

    fc = dataframe_to_ee_points(year_df)
    img = get_annual_embedding_image(year)

    sampled = img.sampleRegions(
        collection=fc,
        properties=[
            'sample_id',
            'unit_id',
            'zone_id',
            'land_cover',
            'observation_year',
            'observation_month',
            'latitude',
            'longitude'
        ],
        scale=SCALE,
        geometries=False,
        tileScale=TILE_SCALE
    )

    out_df = geemap.ee_to_df(sampled)

    print('Returned rows:', len(out_df))

    return out_df

## 3.4 Extract and save yearly caches

In [ ]:
# ------------------------------------------------------------
# Extract embeddings year by year
# ------------------------------------------------------------
all_year_parts = []

for year in sorted(cache_input_df['observation_year'].unique()):
    year_df = cache_input_df[cache_input_df['observation_year'] == year].copy()

    year_cache_csv = os.path.join(
        CACHE_DIR,
        f'gee_embedding_obs_year_{year}.csv'
    )

    if os.path.exists(year_cache_csv):
        print(f'Loading existing yearly cache: {year_cache_csv}')
        year_out_df = pd.read_csv(year_cache_csv)
    else:
        year_out_df = extract_embeddings_for_year(year, year_df)
        year_out_df.to_csv(year_cache_csv, index=False)
        print('Saved:', year_cache_csv)

    all_year_parts.append(year_out_df)

embedding_cache_df = pd.concat(all_year_parts, ignore_index=True)

print('Combined embedding cache shape:', embedding_cache_df.shape)
display(embedding_cache_df.head())

Extracting embeddings for 2020 | points: 1180
Returned rows: 1180
Saved: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/gee_embedding_obs_year_2020.csv
Extracting embeddings for 2021 | points: 675


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Returned rows: 675
Saved: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/gee_embedding_obs_year_2021.csv
Extracting embeddings for 2022 | points: 733


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Returned rows: 733
Saved: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/gee_embedding_obs_year_2022.csv
Extracting embeddings for 2023 | points: 362


...

Returned rows: 362
Saved: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/gee_embedding_obs_year_2023.csv
Extracting embeddings for 2024 | points: 432


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Returned rows: 432
Saved: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/gee_embedding_obs_year_2024.csv
Extracting embeddings for 2025 | points: 324
Returned rows: 324
Saved: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/gee_embedding_obs_year_2025.csv
Combined embedding cache shape: (3706, 72)


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,A00,A01,A02,A03,A04,A05,A06,A07,A08,A09,...,A62,A63,land_cover,latitude,longitude,observation_month,observation_year,sample_id,unit_id,zone_id
0,0.186082,0.071111,-0.130165,0.055363,0.079723,0.113741,0.027128,0.166336,-0.044844,-0.079723,...,0.024606,-0.244152,built_up,-4.336549,20.591859,4,2020,KASAI_000001_2020,KASAI_000001,Ilebo
1,0.166336,0.093564,-0.075356,0.038447,0.079723,0.124567,0.024606,0.130165,-0.055363,-0.059116,...,0.007443,-0.221453,built_up,-4.333445,20.590306,4,2020,KASAI_000002_2020,KASAI_000002,Ilebo
2,0.153787,0.079723,-0.059116,0.055363,0.059116,0.166336,-0.006151,0.186082,-0.015748,-0.084214,...,0.088827,-0.284444,built_up,-4.339163,20.591668,4,2020,KASAI_000003_2020,KASAI_000003,Ilebo
3,0.124567,0.199862,-0.113741,0.071111,0.062991,0.214133,-0.017778,0.228897,-0.027128,-0.055363,...,0.093564,-0.336855,built_up,-4.341256,20.600636,4,2020,KASAI_000004_2020,KASAI_000004,Ilebo
4,0.172795,0.153787,-0.059116,0.012057,0.059116,0.221453,0.029773,0.228897,-0.017778,-0.075356,...,0.108512,-0.310096,built_up,-4.341294,20.594186,4,2020,KASAI_000005_2020,KASAI_000005,Ilebo


## 4.1 Validate metadata and embedding bands

In [ ]:
# ------------------------------------------------------------
# Validate metadata and band order
# ------------------------------------------------------------
metadata_cols = [
    'sample_id',
    'unit_id',
    'zone_id',
    'land_cover',
    'observation_year',
    'observation_month',
    'latitude',
    'longitude'
]

missing_meta = [c for c in metadata_cols if c not in embedding_cache_df.columns]
missing_bands = [b for b in embedding_bands if b not in embedding_cache_df.columns]

if missing_meta:
    raise ValueError('Missing metadata columns from extracted cache: ' + ', '.join(missing_meta))

if missing_bands:
    raise ValueError('Missing embedding bands from extracted cache: ' + ', '.join(missing_bands[:10]))

embedding_cache_df = embedding_cache_df[metadata_cols + embedding_bands].copy()

# Drop duplicated sample IDs if any.
n_before = len(embedding_cache_df)

embedding_cache_df = (
    embedding_cache_df
    .drop_duplicates(subset=['sample_id'], keep='first')
    .reset_index(drop=True)
)

n_after = len(embedding_cache_df)

print('Rows before duplicate removal:', n_before)
print('Rows after duplicate removal:', n_after)

# Check missing embedding values.
embedding_cache_df[embedding_bands] = embedding_cache_df[embedding_bands].apply(
    pd.to_numeric,
    errors='coerce'
)

embedding_cache_df['n_missing_embedding_bands'] = embedding_cache_df[embedding_bands].isna().sum(axis=1)
embedding_cache_df['has_full_embedding'] = embedding_cache_df['n_missing_embedding_bands'].eq(0)

print('Rows with complete embeddings:', int(embedding_cache_df['has_full_embedding'].sum()))
print('Rows with missing embedding values:', int((~embedding_cache_df['has_full_embedding']).sum()))

display(
    embedding_cache_df['n_missing_embedding_bands']
    .value_counts()
    .sort_index()
    .reset_index(name='n_points')
)

Rows before duplicate removal: 3706
Rows after duplicate removal: 3706
Rows with complete embeddings: 3706
Rows with missing embedding values: 0


,n_missing_embedding_bands,n_points
0,0,3706


## 4.2 Save CSV, metadata CSV, and compressed NPZ

In [ ]:
# ------------------------------------------------------------
# Save final model cache
# ------------------------------------------------------------
final_cache_csv = os.path.join(
    CACHE_DIR,
    'Kasai_reference_annual_satellite_embeddings_observation_year.csv'
)

final_cache_npz = os.path.join(
    CACHE_DIR,
    'Kasai_reference_annual_satellite_embeddings_observation_year.npz'
)

metadata_csv = os.path.join(
    CACHE_DIR,
    'Kasai_reference_annual_satellite_embeddings_observation_year_metadata.csv'
)

embedding_cache_df.to_csv(final_cache_csv, index=False)

embedding_cache_df[metadata_cols + ['n_missing_embedding_bands', 'has_full_embedding']].to_csv(
    metadata_csv,
    index=False
)

X_embedding = embedding_cache_df[embedding_bands].to_numpy(dtype=np.float32)
sample_ids = embedding_cache_df['sample_id'].astype(str).to_numpy()
unit_ids = embedding_cache_df['unit_id'].astype(str).to_numpy()
years = embedding_cache_df['observation_year'].astype(int).to_numpy()
land_cover = embedding_cache_df['land_cover'].astype(str).to_numpy()

np.savez_compressed(
    final_cache_npz,
    X_embedding=X_embedding,
    sample_ids=sample_ids,
    unit_ids=unit_ids,
    years=years,
    land_cover=land_cover,
    embedding_bands=np.array(embedding_bands)
)

print('Saved final CSV cache:')
print(final_cache_csv)

print('Saved metadata CSV:')
print(metadata_csv)

print('Saved compressed NPZ cache:')
print(final_cache_npz)

print('X_embedding shape:', X_embedding.shape)

Saved final CSV cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_annual_satellite_embeddings_observation_year.csv
Saved metadata CSV:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_annual_satellite_embeddings_observation_year_metadata.csv
Saved compressed NPZ cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_annual_satellite_embeddings_observation_year.npz
X_embedding shape: (3706, 64)


# GSE patches

In [ ]:
# ============================================================
# 6 — Create CNN patch caches from annual satellite embeddings
# ============================================================
#
# Creates patch caches for:
#   3 x 3 x 64
#   4 x 4 x 64
#   6 x 6 x 64
#   8 x 8 x 64
#   10 x 10 x 64
#   12 x 12 x 64
#
# Output files in CACHE_DIR:
#   Kasai_reference_gee_embedding_patch_p3_observation_year.npz
#   Kasai_reference_gee_embedding_patch_p4_observation_year.npz
#   ...
#   Kasai_reference_gee_embedding_patch_p12_observation_year.npz
#
# Notes:
#   - Odd patches are naturally centred on the sampled pixel.
#   - Even patches are slightly asymmetric because there is no exact centre pixel.
#   - For even patches, the sampled pixel is placed just left/up of the geometric centre.
# ============================================================

import os
import ast
import math
import numpy as np
import pandas as pd

try:
    import geemap
except ImportError:
    !pip -q install geemap
    import geemap


# ------------------------------------------------------------
# 6.1 Settings
# ------------------------------------------------------------

PATCH_SIZE_LIST = [3, 5, 7, 9, 11, 13]

PATCH_NODATA_VALUE = -9999.0
PATCH_CHUNK_SIZE = 50

print("Patch sizes to extract:", PATCH_SIZE_LIST)
print("CACHE_DIR:", CACHE_DIR)


# ------------------------------------------------------------
# 6.2 Reuse reference dataframe from annual embedding cache section
# ------------------------------------------------------------

if "cache_input_df" in globals():
    patch_input_df = cache_input_df.copy()
elif "ref_df" in globals():
    patch_input_df = ref_df.copy()
else:
    raise ValueError(
        "Could not find cache_input_df or ref_df. "
        "Run the annual embedding cache section first."
    )

required_cols = [
    "unit_id",
    "latitude",
    "longitude",
    "observation_year"
]

missing_cols = [c for c in required_cols if c not in patch_input_df.columns]

if missing_cols:
    raise ValueError(
        "Missing required columns for patch cache: "
        + ", ".join(missing_cols)
    )

patch_input_df["unit_id"] = patch_input_df["unit_id"].astype(str)
patch_input_df["observation_year"] = patch_input_df["observation_year"].astype(int)

if "sample_id" not in patch_input_df.columns:
    patch_input_df["sample_id"] = (
        patch_input_df["unit_id"].astype(str)
        + "_"
        + patch_input_df["observation_year"].astype(str)
    )

if "zone_id" not in patch_input_df.columns:
    patch_input_df["zone_id"] = ""

if "land_cover" not in patch_input_df.columns:
    patch_input_df["land_cover"] = ""

if "observation_month" not in patch_input_df.columns:
    patch_input_df["observation_month"] = ""

patch_input_df = patch_input_df[
    (patch_input_df["observation_year"] >= EMBEDDING_START_YEAR)
    & (patch_input_df["observation_year"] <= EMBEDDING_END_YEAR)
].copy()

patch_input_df = patch_input_df.reset_index(drop=True)

print("Rows for patch extraction:", len(patch_input_df))
print("Years:")
display(
    patch_input_df["observation_year"]
    .value_counts()
    .sort_index()
    .reset_index(name="n")
)

if patch_input_df.empty:
    raise ValueError("No rows available for patch extraction.")


# ------------------------------------------------------------
# 6.3 Helper functions
# ------------------------------------------------------------

def dataframe_to_ee_points_for_patches(df):
    features = []

    for _, row in df.iterrows():
        props = {
            "sample_id": str(row["sample_id"]),
            "unit_id": str(row["unit_id"]),
            "zone_id": str(row.get("zone_id", "")),
            "land_cover": str(row.get("land_cover", "")),
            "observation_year": int(row["observation_year"]),
            "observation_month": str(row.get("observation_month", "")),
            "latitude": float(row["latitude"]),
            "longitude": float(row["longitude"])
        }

        geom = ee.Geometry.Point([
            float(row["longitude"]),
            float(row["latitude"])
        ])

        features.append(ee.Feature(geom, props))

    return ee.FeatureCollection(features)


def get_annual_embedding_image_for_patch(year):
    start = ee.Date.fromYMD(int(year), 1, 1)
    end = start.advance(1, "year")

    img = (
        sat_embedding_annual
        .filterDate(start, end)
        .mosaic()
        .select(embedding_bands)
        .unmask(PATCH_NODATA_VALUE)
    )

    return img


def make_exact_patch_kernel(patch_size):
    """
    Create an exact patch_size x patch_size pixel kernel.

    For odd patch sizes, the kernel is centred on the sampled pixel.
    For even patch sizes, the kernel is necessarily asymmetric by 1 pixel.
    """
    patch_size = int(patch_size)

    if patch_size < 1:
        raise ValueError("patch_size must be >= 1")

    weights = np.ones((patch_size, patch_size), dtype=int).tolist()

    # Anchor pixel location inside the kernel.
    # For 3: anchor = 1 -> centred.
    # For 4: anchor = 1 -> patch spans one pixel before and two after.
    # For 6: anchor = 2 -> patch spans two pixels before and three after.
    anchor = (patch_size - 1) // 2

    kernel = ee.Kernel.fixed(
        width=patch_size,
        height=patch_size,
        weights=weights,
        x=anchor,
        y=anchor,
        normalize=False
    )

    return kernel


def make_patch_image(year, patch_size):
    img = get_annual_embedding_image_for_patch(year)

    kernel = make_exact_patch_kernel(patch_size)

    patch_img = img.neighborhoodToArray(kernel)

    return patch_img


def parse_ee_array_value(x):
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)

    if isinstance(x, list):
        return np.array(x, dtype=np.float32)

    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x), dtype=np.float32)
        except Exception:
            raise ValueError(f"Could not parse array string: {x[:100]}")

    raise ValueError(f"Unexpected EE array value type: {type(x)}")


def dataframe_to_patch_array(df_chunk, patch_size):
    patches = []

    for _, row in df_chunk.iterrows():
        band_arrays = []

        for band in embedding_bands:
            arr = parse_ee_array_value(row[band])

            if arr.shape != (patch_size, patch_size):
                raise ValueError(
                    f"Unexpected patch shape for band {band}: {arr.shape}. "
                    f"Expected {(patch_size, patch_size)}."
                )

            band_arrays.append(arr)

        patch = np.stack(band_arrays, axis=-1).astype(np.float32)
        patches.append(patch)

    return np.stack(patches, axis=0).astype(np.float32)


def extract_patch_chunk(year, df_chunk, patch_size):
    fc = dataframe_to_ee_points_for_patches(df_chunk)
    patch_img = make_patch_image(year, patch_size)

    sampled = patch_img.sampleRegions(
        collection=fc,
        properties=[
            "sample_id",
            "unit_id",
            "zone_id",
            "land_cover",
            "observation_year",
            "observation_month",
            "latitude",
            "longitude"
        ],
        scale=SCALE,
        geometries=False,
        tileScale=TILE_SCALE
    )

    sampled_df = geemap.ee_to_df(sampled)

    if sampled_df.empty:
        raise ValueError(f"No samples returned for year {year} chunk.")

    sampled_df["sample_id"] = sampled_df["sample_id"].astype(str)

    requested_order = df_chunk["sample_id"].astype(str).tolist()

    sampled_df["_order"] = sampled_df["sample_id"].map(
        {sid: i for i, sid in enumerate(requested_order)}
    )

    sampled_df = (
        sampled_df
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    missing_returned = set(requested_order) - set(sampled_df["sample_id"].astype(str))

    if len(missing_returned) > 0:
        raise ValueError(
            f"Some requested samples were not returned for year {year}. "
            f"First missing: {list(missing_returned)[:10]}"
        )

    X_chunk = dataframe_to_patch_array(sampled_df, patch_size)

    meta_cols = [
        "sample_id",
        "unit_id",
        "zone_id",
        "land_cover",
        "observation_year",
        "observation_month",
        "latitude",
        "longitude"
    ]

    meta_chunk = sampled_df[meta_cols].copy()

    return X_chunk, meta_chunk


# ------------------------------------------------------------
# 6.4 Main extraction function for one patch size
# ------------------------------------------------------------

def build_patch_cache_for_size(patch_size):
    patch_size = int(patch_size)

    patch_cache_basename = (
        f"Kasai_reference_gee_embedding_patch_p{patch_size}_observation_year"
    )

    patch_npz = os.path.join(
        CACHE_DIR,
        patch_cache_basename + ".npz"
    )

    patch_metadata_csv = os.path.join(
        CACHE_DIR,
        patch_cache_basename + "_metadata.csv"
    )

    print("\n" + "=" * 80)
    print(f"Building patch cache: {patch_size} x {patch_size}")
    print("=" * 80)
    print("Output NPZ:", patch_npz)
    print("Output metadata:", patch_metadata_csv)

    if os.path.exists(patch_npz) and os.path.exists(patch_metadata_csv):
        print("Final patch cache already exists. Loading for inspection.")
        data = np.load(patch_npz, allow_pickle=True)
        meta_df = pd.read_csv(patch_metadata_csv)
        print("X_patch shape:", data["X_patch"].shape)
        print("Metadata shape:", meta_df.shape)
        return data["X_patch"], meta_df, patch_npz, patch_metadata_csv

    all_X_parts = []
    all_meta_parts = []

    for year in sorted(patch_input_df["observation_year"].unique()):
        year_df = patch_input_df[
            patch_input_df["observation_year"] == year
        ].copy().reset_index(drop=True)

        year_npz = os.path.join(
            CACHE_DIR,
            f"{patch_cache_basename}_{year}.npz"
        )

        year_meta_csv = os.path.join(
            CACHE_DIR,
            f"{patch_cache_basename}_{year}_metadata.csv"
        )

        if os.path.exists(year_npz) and os.path.exists(year_meta_csv):
            print(f"\nLoading existing yearly patch cache: {year} | p{patch_size}")
            year_data = np.load(year_npz, allow_pickle=True)
            X_year = year_data["X_patch"]
            meta_year = pd.read_csv(year_meta_csv)

        else:
            print(f"\nExtracting patches for {year} | p{patch_size} | points: {len(year_df)}")

            X_chunks = []
            meta_chunks = []

            n_chunks = math.ceil(len(year_df) / PATCH_CHUNK_SIZE)

            for chunk_i in range(n_chunks):
                start_i = chunk_i * PATCH_CHUNK_SIZE
                end_i = min((chunk_i + 1) * PATCH_CHUNK_SIZE, len(year_df))

                df_chunk = year_df.iloc[start_i:end_i].copy()

                print(
                    f"  Year {year} | p{patch_size} "
                    f"| chunk {chunk_i + 1}/{n_chunks} "
                    f"| rows {start_i}:{end_i}"
                )

                X_chunk, meta_chunk = extract_patch_chunk(
                    year=year,
                    df_chunk=df_chunk,
                    patch_size=patch_size
                )

                print("    X_chunk:", X_chunk.shape)

                X_chunks.append(X_chunk)
                meta_chunks.append(meta_chunk)

            X_year = np.concatenate(X_chunks, axis=0)
            meta_year = pd.concat(meta_chunks, ignore_index=True)

            np.savez_compressed(
                year_npz,
                X_patch=X_year,
                sample_ids=meta_year["sample_id"].astype(str).to_numpy(),
                unit_ids=meta_year["unit_id"].astype(str).to_numpy(),
                years=meta_year["observation_year"].astype(int).to_numpy(),
                embedding_bands=np.array(embedding_bands),
                patch_size=np.array([patch_size]),
                nodata_value=np.array([PATCH_NODATA_VALUE])
            )

            meta_year.to_csv(year_meta_csv, index=False)

            print("  Saved yearly NPZ:", year_npz)
            print("  Saved yearly metadata:", year_meta_csv)

        print(f"Year {year} | p{patch_size} X shape:", X_year.shape)

        all_X_parts.append(X_year)
        all_meta_parts.append(meta_year)

    X_patch = np.concatenate(all_X_parts, axis=0).astype(np.float32)
    patch_metadata_df = pd.concat(all_meta_parts, ignore_index=True)

    sort_order = np.argsort(patch_metadata_df["sample_id"].astype(str).values)

    X_patch = X_patch[sort_order]
    patch_metadata_df = patch_metadata_df.iloc[sort_order].reset_index(drop=True)

    patch_metadata_df["n_nodata_values"] = (
        X_patch == PATCH_NODATA_VALUE
    ).reshape(X_patch.shape[0], -1).sum(axis=1)

    patch_metadata_df["has_full_patch"] = patch_metadata_df["n_nodata_values"].eq(0)

    np.savez_compressed(
        patch_npz,
        X_patch=X_patch,
        sample_ids=patch_metadata_df["sample_id"].astype(str).to_numpy(),
        unit_ids=patch_metadata_df["unit_id"].astype(str).to_numpy(),
        years=patch_metadata_df["observation_year"].astype(int).to_numpy(),
        land_cover=patch_metadata_df["land_cover"].astype(str).to_numpy(),
        zone_id=patch_metadata_df["zone_id"].astype(str).to_numpy(),
        latitude=patch_metadata_df["latitude"].astype(float).to_numpy(),
        longitude=patch_metadata_df["longitude"].astype(float).to_numpy(),
        embedding_bands=np.array(embedding_bands),
        patch_size=np.array([patch_size]),
        nodata_value=np.array([PATCH_NODATA_VALUE])
    )

    patch_metadata_df.to_csv(patch_metadata_csv, index=False)

    print("\nFinal patch cache saved:")
    print(patch_npz)
    print(patch_metadata_csv)

    print("\nX_patch shape:", X_patch.shape)
    print(
        "Expected shape: n_samples x",
        patch_size,
        "x",
        patch_size,
        "x",
        len(embedding_bands)
    )

    print("\nPatch completeness:")
    display(
        patch_metadata_df["has_full_patch"]
        .value_counts()
        .reset_index(name="n")
        .rename(columns={"index": "has_full_patch"})
    )

    print("\nFirst rows of metadata:")
    display(patch_metadata_df.head())

    return X_patch, patch_metadata_df, patch_npz, patch_metadata_csv


# ------------------------------------------------------------
# 6.5 Build all requested patch caches
# ------------------------------------------------------------

patch_cache_outputs = {}

for patch_size in PATCH_SIZE_LIST:
    X_patch, patch_meta, patch_npz, patch_metadata_csv = build_patch_cache_for_size(
        patch_size
    )

    patch_cache_outputs[f"p{patch_size}"] = {
        "X_patch": X_patch,
        "metadata": patch_meta,
        "npz": patch_npz,
        "metadata_csv": patch_metadata_csv
    }

print("\nDone. Patch caches created:")

for key, value in patch_cache_outputs.items():
    print(key)
    print("  X_patch:", value["X_patch"].shape)
    print("  NPZ:", value["npz"])
    print("  metadata:", value["metadata_csv"])

Patch sizes to extract: [3, 5, 7, 9, 11, 13]
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year
Rows for patch extraction: 3706
Years:


,observation_year,n
0,2020,1180
1,2021,675
2,2022,733
3,2023,362
4,2024,432
5,2025,324



Building patch cache: 3 x 3
Output NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p3_observation_year.npz
Output metadata: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p3_observation_year_metadata.csv

Extracting patches for 2020 | p3 | points: 1180
  Year 2020 | p3 | chunk 1/6 | rows 0:200


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


    X_chunk: (200, 3, 3, 64)
  Year 2020 | p3 | chunk 2/6 | rows 200:400


...

    X_chunk: (124, 13, 13, 64)
  Saved yearly NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p13_observation_year_2025.npz
  Saved yearly metadata: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p13_observation_year_2025_metadata.csv
Year 2025 | p13 X shape: (324, 13, 13, 64)

Final patch cache saved:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p13_observation_year.npz
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p13_observation_year_metadata.csv

X_patch shape: (3706, 13, 13, 64)
Expected shape: n_samples x 13 x 13 x 64

Patch completeness:


,has_full_patch,n
0,True,3706



First rows of metadata:


,sample_id,unit_id,zone_id,land_cover,observation_year,observation_month,latitude,longitude,n_nodata_values,has_full_patch
0,KASAI_000001_2020,KASAI_000001,Ilebo,built_up,2020,4,-4.336549,20.591859,0,True
1,KASAI_000002_2020,KASAI_000002,Ilebo,built_up,2020,4,-4.333445,20.590306,0,True
2,KASAI_000003_2020,KASAI_000003,Ilebo,built_up,2020,4,-4.339163,20.591668,0,True
3,KASAI_000004_2020,KASAI_000004,Ilebo,built_up,2020,4,-4.341256,20.600636,0,True
4,KASAI_000005_2020,KASAI_000005,Ilebo,built_up,2020,4,-4.341294,20.594186,0,True



Done. Patch caches created:
p3
  X_patch: (3706, 3, 3, 64)
  NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p3_observation_year.npz
  metadata: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p3_observation_year_metadata.csv
p5
  X_patch: (3706, 5, 5, 64)
  NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p5_observation_year.npz
  metadata: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p5_observation_year_metadata.csv
p7
  X_patch: (3706, 7, 7, 64)
  NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year/Kasai_reference_gee_embedding_patch_p7_observation_year.npz


In [ ]:
# ============================================================
# Move GEE satellite embedding patch caches to separate folder
# ============================================================

import os
import shutil
from pathlib import Path

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN_cropland"

if "CACHE_DIR" not in globals():
    # This is the old/current folder where the patch files were saved.
    CACHE_DIR = os.path.join(
        PROJECT_ROOT,
        "Data",
        "Cache",
        "annual_satellite_embeddings"
    )

SOURCE_DIR = Path(CACHE_DIR)

PATCH_CACHE_DIR = Path(
    PROJECT_ROOT
) / "Data" / "Cache" / "annual_satellite_embeddings_patches"

PATCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Source folder:")
print(SOURCE_DIR)

print("\nTarget patch folder:")
print(PATCH_CACHE_DIR)


# ------------------------------------------------------------
# Find patch cache files
# ------------------------------------------------------------

patch_patterns = [
    "Kasai_reference_gee_embedding_patch_p*_observation_year*.npz",
    "Kasai_reference_gee_embedding_patch_p*_observation_year*.csv",
]

patch_files = []

for pattern in patch_patterns:
    patch_files.extend(sorted(SOURCE_DIR.glob(pattern)))

# Remove duplicates just in case.
patch_files = sorted(set(patch_files))

print("\nPatch files found:", len(patch_files))

for f in patch_files[:20]:
    print(" ", f.name)

if len(patch_files) > 20:
    print(" ...")


# ------------------------------------------------------------
# Move files safely
# ------------------------------------------------------------

moved_files = []
skipped_files = []

for src in patch_files:
    dst = PATCH_CACHE_DIR / src.name

    if dst.exists():
        # If same file already exists in target, skip instead of overwriting.
        skipped_files.append((src, dst))
        continue

    shutil.move(str(src), str(dst))
    moved_files.append((src, dst))

print("\nMoved files:", len(moved_files))
print("Skipped because already present in target:", len(skipped_files))

if moved_files:
    print("\nMoved:")
    for src, dst in moved_files[:30]:
        print(f"  {src.name}  ->  {dst}")

if skipped_files:
    print("\nSkipped:")
    for src, dst in skipped_files[:30]:
        print(f"  {src.name} already exists in target")


# ------------------------------------------------------------
# Update CACHE variables for later CNN cells
# ------------------------------------------------------------

PATCH_CACHE_DIR = str(PATCH_CACHE_DIR)

print("\nUse this folder for CNN patch modelling:")
print("PATCH_CACHE_DIR =", PATCH_CACHE_DIR)

print("\nDone.")

Source folder:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_obs_year

Target patch folder:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/annual_satellite_embeddings_patches

Patch files found: 84
  Kasai_reference_gee_embedding_patch_p11_observation_year.npz
  Kasai_reference_gee_embedding_patch_p11_observation_year_2020.npz
  Kasai_reference_gee_embedding_patch_p11_observation_year_2020_metadata.csv
  Kasai_reference_gee_embedding_patch_p11_observation_year_2021.npz
  Kasai_reference_gee_embedding_patch_p11_observation_year_2021_metadata.csv
  Kasai_reference_gee_embedding_patch_p11_observation_year_2022.npz
  Kasai_reference_gee_embedding_patch_p11_observation_year_2022_metadata.csv
  Kasai_reference_gee_embedding_patch_p11_observation_year_2023.npz
  Kasai_reference_gee_embedding_patch_p11_observation_year_2023_metadata.csv
  Kasai_reference_gee_embedding_patch_p11_observation_year_2024.npz
  Kasai_reference_gee_em

# S2 Temporal

In [ ]:
# ============================================================
# Create Sentinel-2 temporal predictor caches
# T12 = 12 monthly predictors for observation year
# T24 = previous year + observation year
#
# Output folder:
#   Data/Cache/Sentinel2_temporal
#
# Outputs:
#   Kasai_reference_sentinel2_temporal_T12_observation_year.npz
#   Kasai_reference_sentinel2_temporal_T12_observation_year_metadata.csv
#   Kasai_reference_sentinel2_temporal_T24_prev_plus_observation_year.npz
#   Kasai_reference_sentinel2_temporal_T24_prev_plus_observation_year_metadata.csv
#
# Important fix:
#   Earth Engine does not accept np.nan in ee.Image.constant().
#   Missing months are represented by fully masked images instead.
# ============================================================

import os
import time
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# Earth Engine auth/init
# ------------------------------------------------------------
import ee
try:
    ee.Initialize(project='panafricanlu')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='panafricanlu')
print('Earth Engine initialized.')


# ============================================================
# 1. Settings
# ============================================================

# Your simplified NN project folder
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN"

DATA_DIR = os.path.join(PROJECT_ROOT, "Data")

REFERENCE_CSV = os.path.join(DATA_DIR, "Kasai_reference.csv")

S2_CACHE_DIR = os.path.join(
    DATA_DIR,
    "Cache",
    "Sentinel2_temporal"
)

os.makedirs(S2_CACHE_DIR, exist_ok=True)

# Lower this if you hit quota / timeout issues.
CHUNK_SIZE = 150

# Cloud probability threshold.
CLOUD_PROB_THRESHOLD = 40

# Minimum valid observations for monthly value quality summaries.
MIN_VALID_OBS = 1

# Sentinel-2 reflectance bands.
S2_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]

# Spectral indices.
INDEX_NAMES = ["NDVI", "EVI", "NDWI", "NBR"]

# Final feature order in temporal tensor.
FEATURE_NAMES = (
    S2_BANDS
    + INDEX_NAMES
    + ["valid_obs_count"]
)

# Extraction scale.
S2_SCALE = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REFERENCE_CSV:", REFERENCE_CSV)
print("S2_CACHE_DIR:", S2_CACHE_DIR)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("CLOUD_PROB_THRESHOLD:", CLOUD_PROB_THRESHOLD)
print("FEATURE_NAMES:", FEATURE_NAMES)


# ============================================================
# 2. Read reference points
# ============================================================

if not os.path.exists(REFERENCE_CSV):
    raise FileNotFoundError(f"Could not find reference CSV:\n{REFERENCE_CSV}")

ref_df = pd.read_csv(REFERENCE_CSV)

print("\nLoaded reference CSV:", ref_df.shape)
print("Columns:", list(ref_df.columns))


def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


unit_col = find_col(ref_df, ["unit_id", "id", "sample_id"])
lc_col = find_col(ref_df, ["land_cover", "landcover", "label", "class"])
year_col = find_col(ref_df, ["observation_year", "obs_year", "year"])
month_col = find_col(ref_df, ["observation_month", "obs_month", "month"])
lat_col = find_col(ref_df, ["latitude", "lat", "y"])
lon_col = find_col(ref_df, ["longitude", "lon", "long", "x"])
zone_col = find_col(ref_df, ["zone_id", "zone", "site"])

required = {
    "unit_id": unit_col,
    "land_cover": lc_col,
    "observation_year": year_col,
    "observation_month": month_col,
    "latitude": lat_col,
    "longitude": lon_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        "Could not detect required columns: "
        + ", ".join(missing)
        + f"\nAvailable columns: {list(ref_df.columns)}"
    )

rename_dict = {
    unit_col: "unit_id",
    lc_col: "land_cover",
    year_col: "observation_year",
    month_col: "observation_month",
    lat_col: "latitude",
    lon_col: "longitude"
}

if zone_col is not None:
    rename_dict[zone_col] = "zone_id"

ref_df = ref_df.rename(columns=rename_dict).copy()

if "zone_id" not in ref_df.columns:
    ref_df["zone_id"] = ""

ref_df["unit_id"] = ref_df["unit_id"].astype(str)
ref_df["land_cover"] = ref_df["land_cover"].astype(str)
ref_df["zone_id"] = ref_df["zone_id"].astype(str)
ref_df["observation_year"] = ref_df["observation_year"].astype(int)
ref_df["observation_month"] = ref_df["observation_month"].astype(int)
ref_df["latitude"] = pd.to_numeric(ref_df["latitude"], errors="coerce")
ref_df["longitude"] = pd.to_numeric(ref_df["longitude"], errors="coerce")

ref_df = ref_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

# Create sample_id if absent.
# This keeps multiple observations from same unit_id distinct if needed.
if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(str)
        + "_"
        + ref_df["observation_month"].astype(str).str.zfill(2)
    )
else:
    ref_df["sample_id"] = ref_df["sample_id"].astype(str)

# Drop exact duplicate sample IDs.
n_before = len(ref_df)
ref_df = (
    ref_df
    .drop_duplicates(subset=["sample_id"], keep="first")
    .reset_index(drop=True)
)
n_after = len(ref_df)

print("Rows before duplicate sample_id removal:", n_before)
print("Rows after duplicate sample_id removal:", n_after)

display(ref_df.head())

print("\nLand-cover counts:")
display(ref_df["land_cover"].value_counts().reset_index(name="n"))

print("\nObservation year counts:")
display(ref_df["observation_year"].value_counts().sort_index().reset_index(name="n"))


# ============================================================
# 3. Sentinel-2 data sources
# ============================================================

s2_sr = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
s2_clouds = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")


# ============================================================
# 4. Sentinel-2 preparation functions
# ============================================================

def mask_s2_sr_clouds_and_add_features(joined_feature):
    """
    Prepare one joined S2 SR + S2 cloud probability image.

    Output bands:
      B2, B3, B4, B8, B11, B12,
      NDVI, EVI, NDWI, NBR

    Reflectance is scaled to 0-1.
    """
    joined_feature = ee.Feature(joined_feature)
    img = ee.Image(joined_feature.get("primary"))
    cloud_prob = ee.Image(joined_feature.get("secondary"))

    scl = img.select("SCL")

    cloud_mask = cloud_prob.select("probability").lt(CLOUD_PROB_THRESHOLD)

    # SCL mask:
    # 3  = cloud shadow
    # 8  = medium probability cloud
    # 9  = high probability cloud
    # 10 = thin cirrus
    # 11 = snow/ice
    scl_mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    mask = cloud_mask.And(scl_mask)

    scaled = (
        img
        .select(S2_BANDS)
        .multiply(0.0001)
        .updateMask(mask)
        .toFloat()
    )

    blue = scaled.select("B2")
    green = scaled.select("B3")
    red = scaled.select("B4")
    nir = scaled.select("B8")
    swir2 = scaled.select("B12")

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1.0))
        .rename("EVI")
    )
    ndwi = green.subtract(nir).divide(green.add(nir)).rename("NDWI")
    nbr = nir.subtract(swir2).divide(nir.add(swir2)).rename("NBR")

    out = scaled.addBands([ndvi, evi, ndwi, nbr]).toFloat()

    return out.copyProperties(img, img.propertyNames())


def make_empty_masked_s2_feature_image():
    """
    Create a fully masked image with expected S2 feature bands.

    This avoids sending np.nan to Earth Engine, which causes:
      Invalid JSON payload received. Unexpected token NaN
    """
    empty = (
        ee.Image.constant([0] * (len(S2_BANDS) + len(INDEX_NAMES)))
        .rename(S2_BANDS + INDEX_NAMES)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )
    return empty


def make_empty_valid_count_image():
    """
    Count image for months with no valid imagery.
    Count should be 0, not masked.
    """
    return ee.Image.constant(0).rename("valid_obs_count").toFloat()


def build_s2_monthly_composite(region, year, month):
    """
    Monthly Sentinel-2 median composite + valid observation count.

    Output bands:
      B2, B3, B4, B8, B11, B12,
      NDVI, EVI, NDWI, NBR,
      valid_obs_count
    """
    start = ee.Date.fromYMD(int(year), int(month), 1)
    end = start.advance(1, "month")

    sr = (
        s2_sr
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 85))
    )

    clouds = (
        s2_clouds
        .filterBounds(region)
        .filterDate(start, end)
    )

    joined = ee.Join.inner().apply(
        primary=sr,
        secondary=clouds,
        condition=ee.Filter.equals(
            leftField="system:index",
            rightField="system:index"
        )
    )

    prepared = ee.ImageCollection(
        joined.map(mask_s2_sr_clouds_and_add_features)
    )

    empty_features = make_empty_masked_s2_feature_image()
    empty_count = make_empty_valid_count_image()

    composite = ee.Image(
        ee.Algorithms.If(
            prepared.size().gt(0),
            prepared.median().select(S2_BANDS + INDEX_NAMES).toFloat(),
            empty_features
        )
    )

    # Count valid observations per pixel, using NDVI mask after cloud/shadow masking.
    valid_count = ee.Image(
        ee.Algorithms.If(
            prepared.size().gt(0),
            prepared.select("NDVI").count().rename("valid_obs_count").toFloat(),
            empty_count
        )
    )

    return composite.addBands(valid_count).toFloat()


def make_points_fc(df):
    """
    Convert pandas chunk to ee.FeatureCollection.
    """
    features = []

    for _, r in df.iterrows():
        geom = ee.Geometry.Point([float(r["longitude"]), float(r["latitude"])])
        feat = ee.Feature(
            geom,
            {
                "sample_id": str(r["sample_id"]),
                "unit_id": str(r["unit_id"]),
                "zone_id": str(r["zone_id"]),
                "land_cover": str(r["land_cover"]),
                "observation_year": int(r["observation_year"]),
                "observation_month": int(r["observation_month"]),
                "latitude": float(r["latitude"]),
                "longitude": float(r["longitude"]),
            }
        )
        features.append(feat)

    return ee.FeatureCollection(features)


def extract_month_for_points(df_chunk, year, month):
    """
    Extract one monthly S2 composite for a chunk of points.
    Returns dataframe with sample_id + feature columns.
    """
    points_fc = make_points_fc(df_chunk)
    region = points_fc.geometry().bounds().buffer(5000)

    img = build_s2_monthly_composite(region, year, month)

    sampled = img.reduceRegions(
        collection=points_fc,
        reducer=ee.Reducer.first(),
        scale=S2_SCALE,
        tileScale=4
    )

    out = sampled.getInfo()

    rows = []

    for f in out["features"]:
        p = f["properties"]

        row = {
            "sample_id": str(p.get("sample_id")),
        }

        for feat in FEATURE_NAMES:
            v = p.get(feat, None)
            row[feat] = np.nan if v is None else v

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 5. Time-window helpers
# ============================================================

def sequence_months_for_row(obs_year, obs_month, mode):
    """
    Returns list of (year, month) tuples.

    T12:
      Jan-Dec of observation year.

    T24:
      Jan-Dec of previous year + Jan-Dec of observation year.

    Note:
      obs_month is retained in metadata, but here we use full calendar-year phenology.
    """
    obs_year = int(obs_year)

    if mode == "T12":
        return [(obs_year, m) for m in range(1, 13)]

    if mode == "T24":
        return (
            [(obs_year - 1, m) for m in range(1, 13)]
            + [(obs_year, m) for m in range(1, 13)]
        )

    raise ValueError("mode must be 'T12' or 'T24'")


def build_empty_tensor(n_samples, n_steps, n_features):
    return np.full((n_samples, n_steps, n_features), np.nan, dtype=np.float32)


# ============================================================
# 6. Main extraction function
# ============================================================

def build_s2_temporal_cache(ref_df, mode):
    """
    Build Sentinel-2 temporal tensor:
      T12: n_samples x 12 x n_features
      T24: n_samples x 24 x n_features
    """
    if mode not in ["T12", "T24"]:
        raise ValueError("mode must be 'T12' or 'T24'")

    n_steps = 12 if mode == "T12" else 24
    n_features = len(FEATURE_NAMES)
    n_samples = len(ref_df)

    X = build_empty_tensor(n_samples, n_steps, n_features)

    meta = ref_df[
        [
            "sample_id",
            "unit_id",
            "zone_id",
            "land_cover",
            "observation_year",
            "observation_month",
            "latitude",
            "longitude"
        ]
    ].copy()

    meta["n_steps"] = n_steps
    meta["temporal_mode"] = mode

    print("\n" + "=" * 80)
    print(f"Building Sentinel-2 temporal cache: {mode}")
    print("Samples:", n_samples)
    print("Steps:", n_steps)
    print("Features:", n_features, FEATURE_NAMES)
    print("=" * 80)

    extraction_tasks = {}

    for row_idx, r in ref_df.iterrows():
        seq = sequence_months_for_row(
            r["observation_year"],
            r["observation_month"],
            mode
        )

        for step_idx, (yy, mm) in enumerate(seq):
            key = (int(yy), int(mm))
            if key not in extraction_tasks:
                extraction_tasks[key] = []
            extraction_tasks[key].append((row_idx, step_idx))

    print("Unique year-month composites needed:", len(extraction_tasks))
    print("First few:", list(sorted(extraction_tasks.keys()))[:10])

    for task_i, ((yy, mm), row_step_pairs) in enumerate(sorted(extraction_tasks.items()), start=1):
        row_indices = [x[0] for x in row_step_pairs]
        step_indices = [x[1] for x in row_step_pairs]

        task_df = ref_df.iloc[row_indices].copy().reset_index(drop=True)
        task_df["_global_row_index"] = row_indices
        task_df["_step_index"] = step_indices

        print(
            f"\n[{task_i}/{len(extraction_tasks)}] "
            f"Extracting {yy}-{str(mm).zfill(2)} | points: {len(task_df)}"
        )

        n_chunks = int(np.ceil(len(task_df) / CHUNK_SIZE))

        for chunk_i in range(n_chunks):
            start_i = chunk_i * CHUNK_SIZE
            end_i = min((chunk_i + 1) * CHUNK_SIZE, len(task_df))

            chunk = task_df.iloc[start_i:end_i].copy().reset_index(drop=True)

            print(
                f"  chunk {chunk_i + 1}/{n_chunks} | rows {start_i}:{end_i}"
            )

            try:
                extracted = extract_month_for_points(chunk, yy, mm)
            except Exception as e:
                print("  WARNING: extraction failed for this chunk.")
                print("  Year-month:", yy, mm)
                print("  Error:", repr(e))
                print("  Leaving these values as NaN.")
                continue

            extracted = extracted.merge(
                chunk[["sample_id", "_global_row_index", "_step_index"]],
                on="sample_id",
                how="left"
            )

            for _, er in extracted.iterrows():
                if pd.isna(er["_global_row_index"]) or pd.isna(er["_step_index"]):
                    continue

                global_idx = int(er["_global_row_index"])
                step_idx = int(er["_step_index"])

                for feat_idx, feat_name in enumerate(FEATURE_NAMES):
                    val = er.get(feat_name, np.nan)
                    X[global_idx, step_idx, feat_idx] = (
                        np.nan if pd.isna(val) else float(val)
                    )

            # Small pause to reduce hammering GEE.
            time.sleep(0.2)

    # Summary metadata.
    valid_count_idx = FEATURE_NAMES.index("valid_obs_count")

    valid_counts = X[:, :, valid_count_idx]

    meta["n_months_with_valid_obs"] = np.sum(
        valid_counts >= MIN_VALID_OBS,
        axis=1
    ).astype(int)

    meta["n_months_without_valid_obs"] = np.sum(
        valid_counts < MIN_VALID_OBS,
        axis=1
    ).astype(int)

    # Count missing values for all non-count features.
    n_non_count_features = len(S2_BANDS) + len(INDEX_NAMES)
    feature_nan_mask = np.isnan(X[:, :, :n_non_count_features])

    meta["n_missing_feature_values"] = (
        feature_nan_mask
        .reshape(n_samples, -1)
        .sum(axis=1)
        .astype(int)
    )

    meta["has_any_missing_feature_values"] = meta["n_missing_feature_values"].gt(0)
    meta["has_full_temporal_features"] = meta["n_missing_feature_values"].eq(0)

    return X, meta


# ============================================================
# 7. Save function
# ============================================================

def save_s2_cache(X, meta, mode):
    if mode == "T12":
        stem = "Kasai_reference_sentinel2_temporal_T12_observation_year"
    elif mode == "T24":
        stem = "Kasai_reference_sentinel2_temporal_T24_prev_plus_observation_year"
    else:
        raise ValueError("mode must be T12 or T24")

    npz_path = os.path.join(S2_CACHE_DIR, stem + ".npz")
    meta_path = os.path.join(S2_CACHE_DIR, stem + "_metadata.csv")

    meta.to_csv(meta_path, index=False)

    np.savez_compressed(
        npz_path,
        X_s2_temporal=X.astype(np.float32),
        sample_ids=meta["sample_id"].astype(str).to_numpy(),
        unit_ids=meta["unit_id"].astype(str).to_numpy(),
        zone_id=meta["zone_id"].astype(str).to_numpy(),
        land_cover=meta["land_cover"].astype(str).to_numpy(),
        observation_year=meta["observation_year"].astype(int).to_numpy(),
        observation_month=meta["observation_month"].astype(int).to_numpy(),
        latitude=meta["latitude"].astype(float).to_numpy(),
        longitude=meta["longitude"].astype(float).to_numpy(),
        feature_names=np.array(FEATURE_NAMES),
        temporal_mode=np.array([mode]),
        nodata_value=np.array([np.nan])
    )

    print("\nSaved", mode, "cache:")
    print("NPZ:", npz_path)
    print("Metadata:", meta_path)
    print("X shape:", X.shape)

    return npz_path, meta_path


# ============================================================
# 8. Build T12 and T24 caches
# ============================================================

X_s2_T12, meta_T12 = build_s2_temporal_cache(ref_df, mode="T12")
X_s2_T24, meta_T24 = build_s2_temporal_cache(ref_df, mode="T24")


# ============================================================
# 9. Save caches
# ============================================================

t12_npz, t12_meta_csv = save_s2_cache(X_s2_T12, meta_T12, mode="T12")
t24_npz, t24_meta_csv = save_s2_cache(X_s2_T24, meta_T24, mode="T24")


# ============================================================
# 10. Final summaries
# ============================================================

print("\n" + "=" * 80)
print("Sentinel-2 temporal cache summary")
print("=" * 80)

print("\nT12 shape:", X_s2_T12.shape)
display(
    meta_T12[
        [
            "n_months_with_valid_obs",
            "n_months_without_valid_obs",
            "n_missing_feature_values",
            "has_full_temporal_features"
        ]
    ].describe()
)

print("\nT12 valid-month distribution:")
display(
    meta_T12["n_months_with_valid_obs"]
    .value_counts()
    .sort_index()
    .reset_index(name="n_samples")
)

print("\nT24 shape:", X_s2_T24.shape)
display(
    meta_T24[
        [
            "n_months_with_valid_obs",
            "n_months_without_valid_obs",
            "n_missing_feature_values",
            "has_full_temporal_features"
        ]
    ].describe()
)

print("\nT24 valid-month distribution:")
display(
    meta_T24["n_months_with_valid_obs"]
    .value_counts()
    .sort_index()
    .reset_index(name="n_samples")
)

print("\nFiles written:")
print(t12_npz)
print(t12_meta_csv)
print(t24_npz)
print(t24_meta_csv)

print("\nDone.")

Earth Engine initialized.
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
S2_CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal
CHUNK_SIZE: 150
CLOUD_PROB_THRESHOLD: 40
FEATURE_NAMES: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'EVI', 'NDWI', 'NBR', 'valid_obs_count']

Loaded reference CSV: (3706, 9)
Columns: ['unit_id', 'latitude', 'longitude', 'observation_year', 'observation_month', 'zone_id', 'land_cover', 'source_index', 'observation_yearmonth_raw']
Rows before duplicate sample_id removal: 3706
Rows after duplicate sample_id removal: 3706


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020_04
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020_04
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020_04
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020_04
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020_04



Land-cover counts:


,land_cover,n
0,cropland,1039
1,savannah,741
2,water,738
3,forest,598
4,built_up,590



Observation year counts:


...


T24 shape: (3706, 24, 11)


,n_months_with_valid_obs,n_months_without_valid_obs,n_missing_feature_values
count,3706.000000,3706.0,3706.000000
mean,20.909606,0.0,30.903940
std,1.464904,0.0,14.649037
min,15.000000,0.0,0.000000
25%,20.000000,0.0,20.000000
50%,21.000000,0.0,30.000000
75%,22.000000,0.0,40.000000
max,24.000000,0.0,90.000000



T24 valid-month distribution:


,n_months_with_valid_obs,n_samples
0,15,1
1,16,6
2,17,31
3,18,160
4,19,434
5,20,783
6,21,944
7,22,821
8,23,443
9,24,83



Files written:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T12_observation_year.npz
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T12_observation_year_metadata.csv
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T24_prev_plus_observation_year.npz
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T24_prev_plus_observation_year_metadata.csv

Done.


# Sen2 Seasonal metrics

In [ ]:
# ============================================================
# CACHE BLOCK 1 — Sentinel-2 seasonal metrics from existing
# temporal cache
#
# Purpose:
#   Reuse the already created Sentinel-2 temporal cache and
#   derive compact seasonal metrics.
#
# Input:
#   Data/Cache/Sentinel2_temporal/*.npz
#
# Output:
#   Data/Cache/Sentinel2_seasonal_metrics/
#     Kasai_reference_s2_T12_seasonal_metrics.csv
#     Kasai_reference_s2_T12_seasonal_metrics.npz
#     Kasai_reference_s2_T24_seasonal_metrics.csv
#     Kasai_reference_s2_T24_seasonal_metrics.npz
#
# Note:
#   This is not a true spatial patch cache.
#   It is a seasonal metric feature cache derived from the
#   point-based temporal sequence.
# ============================================================

# ------------------------------------------------------------
# Imports and paths
# ------------------------------------------------------------

import os
import re
import glob
import numpy as np
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN"

REFERENCE_CSV = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Kasai_reference.csv"
)

S2_TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Cache",
    "Sentinel2_temporal"
)

S2_SEASONAL_DIR = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Cache",
    "Sentinel2_seasonal_metrics"
)

os.makedirs(S2_SEASONAL_DIR, exist_ok=True)

print("REFERENCE_CSV:", REFERENCE_CSV)
print("S2_TEMPORAL_DIR:", S2_TEMPORAL_DIR)
print("S2_SEASONAL_DIR:", S2_SEASONAL_DIR)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def find_temporal_cache_files(cache_dir):
    files = sorted(glob.glob(os.path.join(cache_dir, "*.npz")))
    if len(files) == 0:
        raise FileNotFoundError(f"No .npz files found in {cache_dir}")
    return files


def infer_cache_label(path):
    name = os.path.basename(path)
    if "T12" in name.upper():
        return "T12"
    if "T24" in name.upper():
        return "T24"
    m = re.search(r"T(\d+)", name.upper())
    if m:
        return f"T{m.group(1)}"
    return os.path.splitext(name)[0]


def load_s2_temporal_npz(path):
    data = np.load(path, allow_pickle=True)
    keys = list(data.keys())
    print("\nLoading:", path)
    print("Keys:", keys)

    # Try common possible array names.
    x_key_candidates = [
        "X_s2_temporal",
        "X_temporal",
        "X_s2",
        "X",
        "X_t12",
        "X_t24"
    ]

    id_key_candidates = [
        "sample_ids",
        "ids",
        "sample_id"
    ]

    feature_key_candidates = [
        "feature_names",
        "features",
        "s2_feature_names",
        "channel_names"
    ]

    x_key = next((k for k in x_key_candidates if k in keys), None)
    id_key = next((k for k in id_key_candidates if k in keys), None)
    feature_key = next((k for k in feature_key_candidates if k in keys), None)

    if x_key is None:
        raise ValueError(
            "Could not find temporal array in NPZ. "
            f"Available keys: {keys}"
        )

    if id_key is None:
        raise ValueError(
            "Could not find sample_ids in NPZ. "
            f"Available keys: {keys}"
        )

    X = data[x_key].astype(np.float32)
    sample_ids = data[id_key].astype(str)

    if feature_key is not None:
        feature_names = data[feature_key].astype(str).tolist()
    else:
        # Fallback to the feature set used in your previous S2 temporal block.
        feature_names = [
            "B2", "B3", "B4", "B8", "B11", "B12",
            "NDVI", "EVI", "NDWI", "NBR",
            "valid_obs_count"
        ]

    if X.ndim != 3:
        raise ValueError(
            f"Expected temporal array with shape (n, time, features), got {X.shape}"
        )

    if X.shape[2] != len(feature_names):
        raise ValueError(
            f"Feature mismatch: X has {X.shape[2]} features, "
            f"but feature_names has {len(feature_names)} entries."
        )

    return X, sample_ids, feature_names


def build_seasonal_metrics(X, feature_names):
    """
    Convert temporal sequence X(n, t, f) into seasonal metrics.

    For spectral/index features:
      median, p10, p90, min, max, std, amplitude

    For valid_obs_count:
      sum, mean, min, max

    Also adds:
      missing_fraction_all_spectral
      n_timesteps
    """

    feature_names = list(feature_names)

    if "valid_obs_count" in feature_names:
        valid_idx = feature_names.index("valid_obs_count")
    else:
        valid_idx = None

    spectral_indices = [
        i for i, f in enumerate(feature_names)
        if f != "valid_obs_count"
    ]

    metric_arrays = []
    metric_names = []

    X_spectral = X[:, :, spectral_indices]

    for local_i, feature_idx in enumerate(spectral_indices):
        fname = feature_names[feature_idx]
        arr = X_spectral[:, :, local_i]

        metrics = {
            "median": np.nanmedian(arr, axis=1),
            "p10": np.nanpercentile(arr, 10, axis=1),
            "p90": np.nanpercentile(arr, 90, axis=1),
            "min": np.nanmin(arr, axis=1),
            "max": np.nanmax(arr, axis=1),
            "std": np.nanstd(arr, axis=1),
        }

        metrics["amplitude"] = metrics["p90"] - metrics["p10"]

        for mname, values in metrics.items():
            metric_arrays.append(values.reshape(-1, 1))
            metric_names.append(f"{fname}_{mname}")

    # Missingness over spectral/index features.
    missing_fraction = np.isnan(X_spectral).mean(axis=(1, 2))
    metric_arrays.append(missing_fraction.reshape(-1, 1))
    metric_names.append("missing_fraction_all_spectral")

    # valid_obs_count summaries.
    if valid_idx is not None:
        valid_arr = X[:, :, valid_idx]

        valid_metrics = {
            "valid_obs_count_sum": np.nansum(valid_arr, axis=1),
            "valid_obs_count_mean": np.nanmean(valid_arr, axis=1),
            "valid_obs_count_min": np.nanmin(valid_arr, axis=1),
            "valid_obs_count_max": np.nanmax(valid_arr, axis=1),
        }

        for mname, values in valid_metrics.items():
            metric_arrays.append(values.reshape(-1, 1))
            metric_names.append(mname)

    n_timesteps = np.full((X.shape[0], 1), X.shape[1], dtype=np.float32)
    metric_arrays.append(n_timesteps)
    metric_names.append("n_timesteps")

    X_metrics = np.hstack(metric_arrays).astype(np.float32)

    # Replace all-NaN metric columns safely.
    col_median = np.nanmedian(X_metrics, axis=0)
    col_median = np.where(np.isfinite(col_median), col_median, 0.0)

    inds = np.where(~np.isfinite(X_metrics))
    X_metrics[inds] = np.take(col_median, inds[1])

    return X_metrics, np.array(metric_names).astype(str)


# ------------------------------------------------------------
# Load reference metadata
# ------------------------------------------------------------

ref_df = pd.read_csv(REFERENCE_CSV)
ref_df["unit_id"] = ref_df["unit_id"].astype(str)

if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(int).astype(str)
        + "_"
        + ref_df["observation_month"].astype(int).astype(str).str.zfill(2)
    )

ref_df["sample_id"] = ref_df["sample_id"].astype(str)

print("Loaded reference:", ref_df.shape)
display(ref_df.head())


# ------------------------------------------------------------
# Build seasonal metric caches for all temporal NPZ files found
# ------------------------------------------------------------

temporal_files = find_temporal_cache_files(S2_TEMPORAL_DIR)
print("Temporal cache files found:")
for f in temporal_files:
    print(" -", f)

for temporal_path in temporal_files:
    cache_label = infer_cache_label(temporal_path)

    X_temporal, sample_ids, feature_names = load_s2_temporal_npz(temporal_path)

    print("Cache label:", cache_label)
    print("X_temporal:", X_temporal.shape)
    print("sample_ids:", sample_ids.shape)
    print("feature_names:", feature_names)

    X_metrics, metric_names = build_seasonal_metrics(
        X_temporal,
        feature_names
    )

    print("X_metrics:", X_metrics.shape)
    print("metric_names:", metric_names.shape)

    metric_df = pd.DataFrame(
        X_metrics,
        columns=metric_names
    )

    metric_df.insert(0, "sample_id", sample_ids)

    metric_df = metric_df.merge(
        ref_df[
            [
                "sample_id",
                "unit_id",
                "zone_id",
                "land_cover",
                "observation_year",
                "observation_month",
                "latitude",
                "longitude"
            ]
        ],
        on="sample_id",
        how="left"
    )

    # Put metadata first.
    meta_cols = [
        "sample_id",
        "unit_id",
        "zone_id",
        "land_cover",
        "observation_year",
        "observation_month",
        "latitude",
        "longitude"
    ]

    metric_df = metric_df[meta_cols + metric_names.tolist()].copy()

    out_csv = os.path.join(
        S2_SEASONAL_DIR,
        f"Kasai_reference_s2_{cache_label}_seasonal_metrics.csv"
    )

    out_npz = os.path.join(
        S2_SEASONAL_DIR,
        f"Kasai_reference_s2_{cache_label}_seasonal_metrics.npz"
    )

    metric_df.to_csv(out_csv, index=False)

    np.savez_compressed(
        out_npz,
        X_metrics=X_metrics.astype(np.float32),
        sample_ids=sample_ids.astype(str),
        metric_names=metric_names.astype(str),
        feature_names=np.array(feature_names).astype(str),
        unit_ids=metric_df["unit_id"].astype(str).to_numpy(),
        zone_id=metric_df["zone_id"].astype(str).to_numpy(),
        land_cover=metric_df["land_cover"].astype(str).to_numpy(),
        observation_year=metric_df["observation_year"].astype(int).to_numpy(),
        observation_month=metric_df["observation_month"].astype(int).to_numpy(),
    )

    print("Saved CSV:", out_csv)
    print("Saved NPZ:", out_npz)

print("Done: Sentinel-2 seasonal metric caches created.")

REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
S2_TEMPORAL_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal
S2_SEASONAL_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_seasonal_metrics
Loaded reference: (3706, 10)


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020_04
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020_04
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020_04
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020_04
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020_04


Temporal cache files found:
 - /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T12_observation_year.npz
 - /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T24_prev_plus_observation_year.npz

Loading: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_temporal/Kasai_reference_sentinel2_temporal_T12_observation_year.npz
Keys: ['X_s2_temporal', 'sample_ids', 'unit_ids', 'zone_id', 'land_cover', 'observation_year', 'observation_month', 'latitude', 'longitude', 'feature_names', 'temporal_mode', 'nodata_value']
Cache label: T12
X_temporal: (3706, 12, 11)
sample_ids: (3706,)
feature_names: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'EVI', 'NDWI', 'NBR', 'valid_obs_count']
X_metrics: (3706, 76)
metric_names: (76,)
Saved CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_seasonal_metrics/Kasai_r

# SEN1 seasonal/annual patch

In [ ]:
# ============================================================
# CACHE BLOCK 2 — Sentinel-1 annual/seasonal SAR patch cache
#
# Purpose:
#   Create spatial context patches from Sentinel-1 around each
#   reference centroid.
#
# Output:
#   Data/Cache/Sentinel1_patches/
#     Kasai_reference_s1_patch_p7_observation_year.npz
#
# Channels:
#   VV_median, VH_median,
#   VV_p10, VV_p90, VV_amplitude,
#   VH_p10, VH_p90, VH_amplitude,
#   VV_minus_VH_median,
#   VV_stdDev, VH_stdDev
# ============================================================

# ------------------------------------------------------------
# Earth Engine auth/init
# ------------------------------------------------------------

import ee
try:
    ee.Initialize(project="panafricanlu")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="panafricanlu")

print("Earth Engine initialized.")


# ------------------------------------------------------------
# Imports and paths
# ------------------------------------------------------------

import os
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN"

REFERENCE_CSV = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Kasai_reference.csv"
)

S1_PATCH_DIR = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Cache",
    "Sentinel1_patches"
)

os.makedirs(S1_PATCH_DIR, exist_ok=True)

PATCH_SIZE = 7
CHUNK_SIZE = 120
S1_SCALE = 10
FILL_VALUE = -9999.0
OVERWRITE = False

OUT_NPZ = os.path.join(
    S1_PATCH_DIR,
    f"Kasai_reference_s1_patch_p{PATCH_SIZE}_observation_year.npz"
)

print("REFERENCE_CSV:", REFERENCE_CSV)
print("S1_PATCH_DIR:", S1_PATCH_DIR)
print("OUT_NPZ:", OUT_NPZ)


# ------------------------------------------------------------
# Load reference CSV
# ------------------------------------------------------------

ref_df = pd.read_csv(REFERENCE_CSV)

required_cols = [
    "unit_id",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "zone_id",
    "land_cover"
]

missing = [c for c in required_cols if c not in ref_df.columns]
if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))

ref_df["unit_id"] = ref_df["unit_id"].astype(str)
ref_df["observation_year"] = ref_df["observation_year"].astype(int)
ref_df["observation_month"] = ref_df["observation_month"].astype(int)

if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(str)
        + "_"
        + ref_df["observation_month"].astype(str).str.zfill(2)
    )

ref_df["sample_id"] = ref_df["sample_id"].astype(str)

ref_df = ref_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

print("Reference rows:", len(ref_df))
display(ref_df.head())


# ------------------------------------------------------------
# Sentinel-1 image builders
# ------------------------------------------------------------

s1 = ee.ImageCollection("COPERNICUS/S1_GRD")


def empty_s1_image(channel_names):
    return (
        ee.Image.constant([FILL_VALUE] * len(channel_names))
        .rename(channel_names)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )


S1_CHANNELS = [
    "S1_VV_median",
    "S1_VH_median",
    "S1_VV_p10",
    "S1_VV_p90",
    "S1_VV_amplitude",
    "S1_VH_p10",
    "S1_VH_p90",
    "S1_VH_amplitude",
    "S1_VV_minus_VH_median",
    "S1_VV_stdDev",
    "S1_VH_stdDev"
]


def get_s1_collection_for_year(year):
    start = ee.Date.fromYMD(int(year), 1, 1)
    end = start.advance(1, "year")

    col = (
        s1
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
    )

    return col


def build_s1_patch_image(year):
    col = get_s1_collection_for_year(year)

    def make_image():
        vv = col.select("VV")
        vh = col.select("VH")

        vv_median = vv.median().rename("S1_VV_median")
        vh_median = vh.median().rename("S1_VH_median")

        vv_p10 = vv.reduce(ee.Reducer.percentile([10])).rename("S1_VV_p10")
        vv_p90 = vv.reduce(ee.Reducer.percentile([90])).rename("S1_VV_p90")
        vv_amp = vv_p90.subtract(vv_p10).rename("S1_VV_amplitude")

        vh_p10 = vh.reduce(ee.Reducer.percentile([10])).rename("S1_VH_p10")
        vh_p90 = vh.reduce(ee.Reducer.percentile([90])).rename("S1_VH_p90")
        vh_amp = vh_p90.subtract(vh_p10).rename("S1_VH_amplitude")

        vv_vh = vv_median.subtract(vh_median).rename("S1_VV_minus_VH_median")

        vv_std = vv.reduce(ee.Reducer.stdDev()).rename("S1_VV_stdDev")
        vh_std = vh.reduce(ee.Reducer.stdDev()).rename("S1_VH_stdDev")

        return ee.Image.cat([
            vv_median,
            vh_median,
            vv_p10,
            vv_p90,
            vv_amp,
            vh_p10,
            vh_p90,
            vh_amp,
            vv_vh,
            vv_std,
            vh_std
        ]).toFloat()

    img = ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            make_image(),
            empty_s1_image(S1_CHANNELS)
        )
    )

    return img.select(S1_CHANNELS).toFloat()


def image_to_patch_array(img, patch_size):
    weights = [[1] * patch_size for _ in range(patch_size)]
    focus = patch_size // 2

    kernel = ee.Kernel.fixed(
        width=patch_size,
        height=patch_size,
        weights=weights,
        x=focus,
        y=focus,
        normalize=False
    )

    return img.neighborhoodToArray(
        kernel=kernel,
        defaultValue=FILL_VALUE
    )


# ------------------------------------------------------------
# FeatureCollection helper
# ------------------------------------------------------------

def chunk_to_fc(df):
    features = []

    for _, row in df.iterrows():
        lon = float(row["longitude"])
        lat = float(row["latitude"])

        props = {
            "sample_id": str(row["sample_id"]),
            "unit_id": str(row["unit_id"]),
            "zone_id": str(row["zone_id"]),
            "land_cover": str(row["land_cover"]),
            "observation_year": int(row["observation_year"]),
            "observation_month": int(row["observation_month"]),
        }

        features.append(
            ee.Feature(
                ee.Geometry.Point([lon, lat]),
                props
            )
        )

    return ee.FeatureCollection(features)


# ------------------------------------------------------------
# Extract patches
# ------------------------------------------------------------

if os.path.exists(OUT_NPZ) and not OVERWRITE:
    print("Cache already exists, loading:", OUT_NPZ)

    cache = np.load(OUT_NPZ, allow_pickle=True)
    X_s1_patch = cache["X"]
    s1_sample_ids = cache["sample_ids"]
    s1_channel_names = cache["channel_names"]

    print("X_s1_patch:", X_s1_patch.shape)

else:
    X_list = []
    sample_id_list = []
    unit_id_list = []
    year_list = []
    month_list = []
    zone_list = []
    label_list = []

    years = sorted(ref_df["observation_year"].unique())

    for year in years:
        year_df = ref_df[ref_df["observation_year"] == year].copy()
        arr_img = image_to_patch_array(
            build_s1_patch_image(year),
            PATCH_SIZE
        )

        print(f"\nExtracting Sentinel-1 patches for {year} | points: {len(year_df)}")

        for start in range(0, len(year_df), CHUNK_SIZE):
            end = min(start + CHUNK_SIZE, len(year_df))
            chunk = year_df.iloc[start:end].copy()

            print(f"  chunk rows {start}:{end}")

            fc = chunk_to_fc(chunk)

            sampled = arr_img.sampleRegions(
                collection=fc,
                properties=[
                    "sample_id",
                    "unit_id",
                    "zone_id",
                    "land_cover",
                    "observation_year",
                    "observation_month"
                ],
                scale=S1_SCALE,
                geometries=False,
                tileScale=4
            )

            info = sampled.getInfo()

            for feat in info["features"]:
                props = feat["properties"]

                patch_channels = []
                ok = True

                for band in S1_CHANNELS:
                    if band not in props:
                        ok = False
                        break

                    arr = np.array(props[band], dtype=np.float32)

                    if arr.shape != (PATCH_SIZE, PATCH_SIZE):
                        ok = False
                        break

                    arr[arr == FILL_VALUE] = np.nan
                    patch_channels.append(arr)

                if not ok:
                    continue

                X_list.append(np.stack(patch_channels, axis=-1))
                sample_id_list.append(str(props["sample_id"]))
                unit_id_list.append(str(props["unit_id"]))
                zone_list.append(str(props["zone_id"]))
                label_list.append(str(props["land_cover"]))
                year_list.append(int(props["observation_year"]))
                month_list.append(int(props["observation_month"]))

            time.sleep(0.2)

    if len(X_list) == 0:
        raise ValueError("No Sentinel-1 patches extracted.")

    X_s1_patch = np.stack(X_list).astype(np.float32)

    np.savez_compressed(
        OUT_NPZ,
        X=X_s1_patch,
        sample_ids=np.array(sample_id_list).astype(str),
        unit_ids=np.array(unit_id_list).astype(str),
        zone_id=np.array(zone_list).astype(str),
        land_cover=np.array(label_list).astype(str),
        observation_year=np.array(year_list).astype(int),
        observation_month=np.array(month_list).astype(int),
        channel_names=np.array(S1_CHANNELS).astype(str),
        patch_size=np.array([PATCH_SIZE]).astype(int)
    )

    print("\nSaved:", OUT_NPZ)
    print("X_s1_patch:", X_s1_patch.shape)

print("Done: Sentinel-1 patch cache.")

Earth Engine initialized.
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
S1_PATCH_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel1_patches
OUT_NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel1_patches/Kasai_reference_s1_patch_p7_observation_year.npz
Reference rows: 3706


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020_04
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020_04
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020_04
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020_04
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020_04



Extracting Sentinel-1 patches for 2020 | points: 1180
  chunk rows 0:120
  chunk rows 120:240
  chunk rows 240:360
  chunk rows 360:480
  chunk rows 480:600
  chunk rows 600:720
  chunk rows 720:840
  chunk rows 840:960
  chunk rows 960:1080
  chunk rows 1080:1180

Extracting Sentinel-1 patches for 2021 | points: 675
  chunk rows 0:120
  chunk rows 120:240
  chunk rows 240:360
  chunk rows 360:480
  chunk rows 480:600
  chunk rows 600:675

Extracting Sentinel-1 patches for 2022 | points: 733
  chunk rows 0:120
  chunk rows 120:240
  chunk rows 240:360
  chunk rows 360:480
  chunk rows 480:600
  chunk rows 600:720
  chunk rows 720:733

Extracting Sentinel-1 patches for 2023 | points: 362
  chunk rows 0:120
  chunk rows 120:240
  chunk rows 240:360
  chunk rows 360:362

Extracting Sentinel-1 patches for 2024 | points: 432
  chunk rows 0:120
  chunk rows 120:240
  chunk rows 240:360
  chunk rows 360:432

Extracting Sentinel-1 patches for 2025 | points: 324
  chunk rows 0:120
  chunk rows

# SEN2 annual composite cache (median +NDVI amplitude)

In [ ]:
# ============================================================
# CACHE BLOCK — Sentinel-2 annual patch cache, high-quality mask
#
# Memory-safe version:
#   - Keeps S2 cloud probability masking
#   - Uses S2 SR + S2_CLOUD_PROBABILITY join
#   - Saves each year/chunk separately
#   - Resumes existing chunks
#   - Uses chunk-local region filtering
#   - Uses quality mosaic for annual spectral/index composite
#   - Adds NDVI p10, p90, amplitude, valid observation count
#
# Output:
#   NN/Data/Cache/Sentinel2_annual_patches/
#     chunks_cloudprob_p7/
#       s2_annual_cloudprob_p7_year2020_chunk0000.npz
#       ...
#     Kasai_reference_s2_annual_cloudprob_median_amp_patch_p7_observation_year.npz
# ============================================================


# ------------------------------------------------------------
# 1. Earth Engine auth/init
# ------------------------------------------------------------

import ee

try:
    ee.Initialize(project="panafricanlu")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="panafricanlu")

print("Earth Engine initialized.")


# ------------------------------------------------------------
# 2. Imports and paths
# ------------------------------------------------------------

import os
import time
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN"

REFERENCE_CSV = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Kasai_reference.csv"
)

S2_ANNUAL_PATCH_DIR = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Cache",
    "Sentinel2_annual_patches"
)

PATCH_SIZE = 7
CHUNK_SIZE = 40       # safer than 120 with cloud probability
S2_SCALE = 10
FILL_VALUE = -9999.0

CLOUD_PROB_THRESHOLD = 40
MAX_SCENE_CLOUD_PERCENT = 85

OVERWRITE_CHUNKS = False
OVERWRITE_FINAL = True

CHUNK_DIR = os.path.join(
    S2_ANNUAL_PATCH_DIR,
    f"chunks_cloudprob_p{PATCH_SIZE}"
)

os.makedirs(S2_ANNUAL_PATCH_DIR, exist_ok=True)
os.makedirs(CHUNK_DIR, exist_ok=True)

OUT_NPZ = os.path.join(
    S2_ANNUAL_PATCH_DIR,
    f"Kasai_reference_s2_annual_cloudprob_median_amp_patch_p{PATCH_SIZE}_observation_year.npz"
)

print("REFERENCE_CSV:", REFERENCE_CSV)
print("S2_ANNUAL_PATCH_DIR:", S2_ANNUAL_PATCH_DIR)
print("CHUNK_DIR:", CHUNK_DIR)
print("OUT_NPZ:", OUT_NPZ)
print("PATCH_SIZE:", PATCH_SIZE)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("CLOUD_PROB_THRESHOLD:", CLOUD_PROB_THRESHOLD)


# ------------------------------------------------------------
# 3. Load reference CSV
# ------------------------------------------------------------

ref_df = pd.read_csv(REFERENCE_CSV)

required_cols = [
    "unit_id",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "zone_id",
    "land_cover"
]

missing = [c for c in required_cols if c not in ref_df.columns]
if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))

ref_df["unit_id"] = ref_df["unit_id"].astype(str)
ref_df["observation_year"] = ref_df["observation_year"].astype(int)
ref_df["observation_month"] = ref_df["observation_month"].astype(int)

if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(str)
        + "_"
        + ref_df["observation_month"].astype(str).str.zfill(2)
    )

ref_df["sample_id"] = ref_df["sample_id"].astype(str)
ref_df = ref_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
ref_df["row_index"] = np.arange(len(ref_df))

print("Reference rows:", len(ref_df))
print("Years:", sorted(ref_df["observation_year"].unique()))
display(ref_df.head())
display(ref_df["land_cover"].value_counts().reset_index(name="n"))


# ------------------------------------------------------------
# 4. Sentinel-2 settings
# ------------------------------------------------------------

s2_sr = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
s2_clouds = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

S2_BASE_FEATURES = [
    "B2", "B3", "B4", "B8", "B11", "B12",
    "NDVI", "EVI", "NDWI", "NBR"
]

S2_ANNUAL_CHANNELS = [
    "S2_B2_qm",
    "S2_B3_qm",
    "S2_B4_qm",
    "S2_B8_qm",
    "S2_B11_qm",
    "S2_B12_qm",
    "S2_NDVI_qm",
    "S2_EVI_qm",
    "S2_NDWI_qm",
    "S2_NBR_qm",
    "S2_NDVI_p10",
    "S2_NDVI_p90",
    "S2_NDVI_amplitude",
    "S2_valid_obs_count"
]


# ------------------------------------------------------------
# 5. Sentinel-2 preparation functions
# ------------------------------------------------------------

def empty_s2_annual_image():
    return (
        ee.Image.constant([FILL_VALUE] * len(S2_ANNUAL_CHANNELS))
        .rename(S2_ANNUAL_CHANNELS)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )


def join_s2_with_cloud_probability(sr_col, cloud_col):
    joined = ee.Join.saveFirst("cloud_prob_img").apply(
        primary=sr_col,
        secondary=cloud_col,
        condition=ee.Filter.equals(
            leftField="system:index",
            rightField="system:index"
        )
    )
    return ee.ImageCollection(joined)


def mask_s2_cloudprob_and_scl(img):
    cloud_prob_img = ee.Image(img.get("cloud_prob_img"))
    cloud_prob = cloud_prob_img.select("probability")

    scl = img.select("SCL")

    cloud_prob_mask = cloud_prob.lt(CLOUD_PROB_THRESHOLD)

    scl_mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    clear_score = ee.Image.constant(100).subtract(cloud_prob).rename("clear_score")

    masked = img.updateMask(cloud_prob_mask.And(scl_mask))

    return masked.addBands(clear_score).copyProperties(img, img.propertyNames())


def add_s2_indices(img):
    img = img.toFloat()

    blue = img.select("B2").divide(10000).rename("B2")
    green = img.select("B3").divide(10000).rename("B3")
    red = img.select("B4").divide(10000).rename("B4")
    nir = img.select("B8").divide(10000).rename("B8")
    swir1 = img.select("B11").divide(10000).rename("B11")
    swir2 = img.select("B12").divide(10000).rename("B12")

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")

    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(
            nir
            .add(red.multiply(6))
            .subtract(blue.multiply(7.5))
            .add(1)
        )
        .rename("EVI")
    )

    ndwi = green.subtract(nir).divide(green.add(nir)).rename("NDWI")
    nbr = nir.subtract(swir2).divide(nir.add(swir2)).rename("NBR")

    clear_score = img.select("clear_score")

    return ee.Image.cat([
        blue,
        green,
        red,
        nir,
        swir1,
        swir2,
        ndvi,
        evi,
        ndwi,
        nbr,
        clear_score
    ]).toFloat().copyProperties(img, img.propertyNames())


def get_s2_cloudprob_collection_for_year(year, region):
    start = ee.Date.fromYMD(int(year), 1, 1)
    end = start.advance(1, "year")

    sr_col = (
        s2_sr
        .filterDate(start, end)
        .filterBounds(region)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PERCENT))
    )

    cloud_col = (
        s2_clouds
        .filterDate(start, end)
        .filterBounds(region)
    )

    joined = join_s2_with_cloud_probability(sr_col, cloud_col)

    prepared = (
        joined
        .filter(ee.Filter.notNull(["cloud_prob_img"]))
        .map(mask_s2_cloudprob_and_scl)
        .map(add_s2_indices)
    )

    return prepared


def build_s2_annual_cloudprob_image(year, region):
    col = get_s2_cloudprob_collection_for_year(year, region)

    def make_image():
        # Less memory-heavy than median across all bands.
        qm = (
            col
            .select(S2_BASE_FEATURES + ["clear_score"])
            .qualityMosaic("clear_score")
            .select(S2_BASE_FEATURES)
            .rename([f"S2_{b}_qm" for b in S2_BASE_FEATURES])
        )

        ndvi = col.select("NDVI")

        ndvi_p10 = (
            ndvi
            .reduce(ee.Reducer.percentile([10]))
            .rename("S2_NDVI_p10")
        )

        ndvi_p90 = (
            ndvi
            .reduce(ee.Reducer.percentile([90]))
            .rename("S2_NDVI_p90")
        )

        ndvi_amp = ndvi_p90.subtract(ndvi_p10).rename("S2_NDVI_amplitude")
        valid_count = ndvi.count().rename("S2_valid_obs_count")

        return ee.Image.cat([
            qm,
            ndvi_p10,
            ndvi_p90,
            ndvi_amp,
            valid_count
        ]).toFloat()

    img = ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            make_image(),
            empty_s2_annual_image()
        )
    )

    return img.select(S2_ANNUAL_CHANNELS).toFloat()


def image_to_patch_array(img, patch_size):
    weights = [[1] * patch_size for _ in range(patch_size)]
    focus = patch_size // 2

    kernel = ee.Kernel.fixed(
        width=patch_size,
        height=patch_size,
        weights=weights,
        x=focus,
        y=focus,
        normalize=False
    )

    return img.neighborhoodToArray(
        kernel=kernel,
        defaultValue=FILL_VALUE
    )


# ------------------------------------------------------------
# 6. FeatureCollection helpers
# ------------------------------------------------------------

def chunk_to_fc(df):
    features = []

    for _, row in df.iterrows():
        lon = float(row["longitude"])
        lat = float(row["latitude"])

        props = {
            "sample_id": str(row["sample_id"]),
            "unit_id": str(row["unit_id"]),
            "zone_id": str(row["zone_id"]),
            "land_cover": str(row["land_cover"]),
            "observation_year": int(row["observation_year"]),
            "observation_month": int(row["observation_month"]),
            "row_index": int(row["row_index"])
        }

        features.append(
            ee.Feature(
                ee.Geometry.Point([lon, lat]),
                props
            )
        )

    return ee.FeatureCollection(features)


def chunk_region(df, buffer_m=1000):
    fc = chunk_to_fc(df)
    return fc.geometry().buffer(buffer_m).bounds()


# ------------------------------------------------------------
# 7. Extract one year/chunk
# ------------------------------------------------------------

def extract_one_chunk(year, chunk_df, chunk_id):
    chunk_path = os.path.join(
        CHUNK_DIR,
        f"s2_annual_cloudprob_p{PATCH_SIZE}_year{int(year)}_chunk{chunk_id:04d}.npz"
    )

    if os.path.exists(chunk_path) and not OVERWRITE_CHUNKS:
        print("    exists, skip:", os.path.basename(chunk_path))
        return chunk_path

    region = chunk_region(chunk_df, buffer_m=1200)

    annual_img = build_s2_annual_cloudprob_image(year, region)
    arr_img = image_to_patch_array(annual_img, PATCH_SIZE)

    fc = chunk_to_fc(chunk_df)

    sampled = arr_img.sampleRegions(
        collection=fc,
        properties=[
            "sample_id",
            "unit_id",
            "zone_id",
            "land_cover",
            "observation_year",
            "observation_month",
            "row_index"
        ],
        scale=S2_SCALE,
        geometries=False,
        tileScale=8
    )

    info = sampled.getInfo()

    X_list = []
    sample_id_list = []
    unit_id_list = []
    zone_list = []
    label_list = []
    year_list = []
    month_list = []
    row_index_list = []

    for feat in info["features"]:
        props = feat["properties"]

        patch_channels = []
        ok = True

        for band in S2_ANNUAL_CHANNELS:
            if band not in props:
                ok = False
                break

            arr = np.array(props[band], dtype=np.float32)

            if arr.shape != (PATCH_SIZE, PATCH_SIZE):
                ok = False
                break

            arr[arr == FILL_VALUE] = np.nan
            patch_channels.append(arr)

        if not ok:
            continue

        X_list.append(np.stack(patch_channels, axis=-1))
        sample_id_list.append(str(props["sample_id"]))
        unit_id_list.append(str(props["unit_id"]))
        zone_list.append(str(props["zone_id"]))
        label_list.append(str(props["land_cover"]))
        year_list.append(int(props["observation_year"]))
        month_list.append(int(props["observation_month"]))
        row_index_list.append(int(props["row_index"]))

    if len(X_list) == 0:
        print("    WARNING: no patches returned.")
        return None

    X_chunk = np.stack(X_list).astype(np.float32)

    np.savez_compressed(
        chunk_path,
        X=X_chunk,
        sample_ids=np.array(sample_id_list).astype(str),
        unit_ids=np.array(unit_id_list).astype(str),
        zone_id=np.array(zone_list).astype(str),
        land_cover=np.array(label_list).astype(str),
        observation_year=np.array(year_list).astype(int),
        observation_month=np.array(month_list).astype(int),
        row_index=np.array(row_index_list).astype(int),
        channel_names=np.array(S2_ANNUAL_CHANNELS).astype(str),
        patch_size=np.array([PATCH_SIZE]).astype(int),
        cloud_prob_threshold=np.array([CLOUD_PROB_THRESHOLD]).astype(int)
    )

    print("    saved:", os.path.basename(chunk_path), "X:", X_chunk.shape)

    return chunk_path


# ------------------------------------------------------------
# 8. Extract all chunks, resumable
# ------------------------------------------------------------

if os.path.exists(OUT_NPZ) and not OVERWRITE_FINAL:
    print("Final cache already exists:")
    print(OUT_NPZ)
    print("Set OVERWRITE_FINAL=True if you want to rebuild final file.")

else:
    years = sorted(ref_df["observation_year"].unique())

    for year in years:
        year_df = ref_df[ref_df["observation_year"] == year].copy()
        year_df = year_df.reset_index(drop=True)

        n_chunks = int(np.ceil(len(year_df) / CHUNK_SIZE))

        print("\n" + "=" * 80)
        print(f"Extracting Sentinel-2 annual cloud-prob patches | year {year}")
        print(f"points: {len(year_df)} | chunks: {n_chunks}")
        print("=" * 80)

        for chunk_id, start in enumerate(range(0, len(year_df), CHUNK_SIZE)):
            end = min(start + CHUNK_SIZE, len(year_df))
            chunk_df = year_df.iloc[start:end].copy()

            print(f"  chunk {chunk_id + 1}/{n_chunks} | rows {start}:{end}")

            try:
                extract_one_chunk(year, chunk_df, chunk_id)

            except Exception as e:
                print("    WARNING: chunk failed.")
                print("    year:", year)
                print("    chunk:", chunk_id)
                print("    rows:", start, end)
                print("    error:", repr(e))
                print("    Continuing with next chunk.")

            time.sleep(0.5)

    print("\nExtraction pass finished.")


# ------------------------------------------------------------
# 9. Combine chunk files into final NPZ
# ------------------------------------------------------------

chunk_files = sorted(glob.glob(
    os.path.join(CHUNK_DIR, f"s2_annual_cloudprob_p{PATCH_SIZE}_year*_chunk*.npz")
))

if len(chunk_files) == 0:
    raise ValueError("No chunk files found. Extraction did not produce cache chunks.")

print("Chunk files found:", len(chunk_files))

X_all = []
sample_ids_all = []
unit_ids_all = []
zone_all = []
label_all = []
year_all = []
month_all = []
row_index_all = []

for path in tqdm(chunk_files, desc="Combining chunks"):
    data = np.load(path, allow_pickle=True)

    X_all.append(data["X"].astype(np.float32))
    sample_ids_all.append(data["sample_ids"].astype(str))
    unit_ids_all.append(data["unit_ids"].astype(str))
    zone_all.append(data["zone_id"].astype(str))
    label_all.append(data["land_cover"].astype(str))
    year_all.append(data["observation_year"].astype(int))
    month_all.append(data["observation_month"].astype(int))
    row_index_all.append(data["row_index"].astype(int))

X_all = np.concatenate(X_all, axis=0)
sample_ids_all = np.concatenate(sample_ids_all)
unit_ids_all = np.concatenate(unit_ids_all)
zone_all = np.concatenate(zone_all)
label_all = np.concatenate(label_all)
year_all = np.concatenate(year_all)
month_all = np.concatenate(month_all)
row_index_all = np.concatenate(row_index_all)

# Sort back to original reference CSV order.
order = np.argsort(row_index_all)

X_all = X_all[order]
sample_ids_all = sample_ids_all[order]
unit_ids_all = unit_ids_all[order]
zone_all = zone_all[order]
label_all = label_all[order]
year_all = year_all[order]
month_all = month_all[order]
row_index_all = row_index_all[order]

# Drop duplicated sample IDs if any.
seen = set()
keep_idx = []

for i, sid in enumerate(sample_ids_all):
    if sid not in seen:
        seen.add(sid)
        keep_idx.append(i)

keep_idx = np.array(keep_idx)

X_all = X_all[keep_idx]
sample_ids_all = sample_ids_all[keep_idx]
unit_ids_all = unit_ids_all[keep_idx]
zone_all = zone_all[keep_idx]
label_all = label_all[keep_idx]
year_all = year_all[keep_idx]
month_all = month_all[keep_idx]
row_index_all = row_index_all[keep_idx]

np.savez_compressed(
    OUT_NPZ,
    X=X_all.astype(np.float32),
    sample_ids=sample_ids_all.astype(str),
    unit_ids=unit_ids_all.astype(str),
    zone_id=zone_all.astype(str),
    land_cover=label_all.astype(str),
    observation_year=year_all.astype(int),
    observation_month=month_all.astype(int),
    row_index=row_index_all.astype(int),
    channel_names=np.array(S2_ANNUAL_CHANNELS).astype(str),
    patch_size=np.array([PATCH_SIZE]).astype(int),
    cloud_prob_threshold=np.array([CLOUD_PROB_THRESHOLD]).astype(int)
)

print("\nSaved final Sentinel-2 annual cloud-probability patch cache:")
print(OUT_NPZ)
print("X:", X_all.shape)
print("sample_ids:", sample_ids_all.shape)
print("channels:", len(S2_ANNUAL_CHANNELS), S2_ANNUAL_CHANNELS)

expected = set(ref_df["sample_id"].astype(str))
got = set(sample_ids_all.astype(str))
missing = sorted(expected - got)

print("\nMissing samples compared with reference:")
print("Missing:", len(missing))
print("First missing:", missing[:20])

print("\nNaN diagnostics:")
print("Total NaNs:", int(np.isnan(X_all).sum()))
print("NaNs per channel:")
nan_per_channel = np.isnan(X_all).sum(axis=(0, 1, 2))
for name, n in zip(S2_ANNUAL_CHANNELS, nan_per_channel):
    print(f"  {name}: {int(n)}")

print("Done.")

Earth Engine initialized.
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
S2_ANNUAL_PATCH_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches
CHUNK_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/chunks_cloudprob_p7
OUT_NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/Kasai_reference_s2_annual_cloudprob_median_amp_patch_p7_observation_year.npz
PATCH_SIZE: 7
CHUNK_SIZE: 40
CLOUD_PROB_THRESHOLD: 40
Reference rows: 3706
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id,row_index
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020_04,0
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020_04,1
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020_04,2
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020_04,3
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020_04,4


,land_cover,n
0,cropland,1039
1,savannah,741
2,water,738
3,forest,598
4,built_up,590



Extracting Sentinel-2 annual cloud-prob patches | year 2020
points: 1180 | chunks: 30
  chunk 1/30 | rows 0:40
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0000.npz
  chunk 2/30 | rows 40:80
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0001.npz
  chunk 3/30 | rows 80:120
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0002.npz
  chunk 4/30 | rows 120:160
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0003.npz
  chunk 5/30 | rows 160:200
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0004.npz
  chunk 6/30 | rows 200:240
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0005.npz
  chunk 7/30 | rows 240:280
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0006.npz
  chunk 8/30 | rows 280:320
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0007.npz
  chunk 9/30 | rows 320:360
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0008.npz
  chunk 10/30 | rows 360:400
    exists, skip: s2_annual_cloudprob_p7_year2020_chunk0009.np

Combining chunks:   0%|          | 0/94 [00:00<?, ?it/s]


Saved final Sentinel-2 annual cloud-probability patch cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/Kasai_reference_s2_annual_cloudprob_median_amp_patch_p7_observation_year.npz
X: (3626, 7, 7, 14)
sample_ids: (3626,)
channels: 14 ['S2_B2_qm', 'S2_B3_qm', 'S2_B4_qm', 'S2_B8_qm', 'S2_B11_qm', 'S2_B12_qm', 'S2_NDVI_qm', 'S2_EVI_qm', 'S2_NDWI_qm', 'S2_NBR_qm', 'S2_NDVI_p10', 'S2_NDVI_p90', 'S2_NDVI_amplitude', 'S2_valid_obs_count']

Missing samples compared with reference:
Missing: 80
First missing: ['KASAI_002062_2024_04', 'KASAI_002063_2024_04', 'KASAI_002064_2024_04', 'KASAI_002065_2024_04', 'KASAI_002066_2024_04', 'KASAI_002067_2024_04', 'KASAI_002068_2024_04', 'KASAI_002069_2024_04', 'KASAI_002070_2024_04', 'KASAI_002071_2024_04', 'KASAI_002072_2024_04', 'KASAI_002073_2024_04', 'KASAI_002074_2024_04', 'KASAI_002075_2024_04', 'KASAI_002076_2024_04', 'KASAI_002077_2024_04', 'KASAI_002078_2024_04', 'KASAI_002079_2024_04', 'KASAI_002080

In [ ]:
# ============================================================
# RECOVERY CELL — Recover missing Sentinel-2 annual patch samples
# and rebuild the final cache safely
# ============================================================
#
# Run this AFTER the main S2 annual patch extraction block.
#
# It expects these variables from the main block:
#   REFERENCE_CSV
#   CHUNK_DIR
#   OUT_NPZ
#   PATCH_SIZE
#   CLOUD_PROB_THRESHOLD
#
# It does NOT change existing successful chunks.
# It only adds recovery chunks for missing samples.
# ============================================================

import os
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import ee

try:
    ee.Initialize(project="panafricanlu")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="panafricanlu")

print("Earth Engine initialized.")

# ------------------------------------------------------------
# 1. Recovery settings
# ------------------------------------------------------------

RECOVERY_CHUNK_SIZE = 1
RECOVERY_TILE_SCALE = 8
RECOVERY_SLEEP_SECONDS = 0.5

# Keep names exactly compatible with your existing final cache.
S2_FEATURE_NAMES = [
    "S2_B2_qm",
    "S2_B3_qm",
    "S2_B4_qm",
    "S2_B8_qm",
    "S2_B11_qm",
    "S2_B12_qm",
    "S2_NDVI_qm",
    "S2_EVI_qm",
    "S2_NDWI_qm",
    "S2_NBR_qm",
    "S2_NDVI_p10",
    "S2_NDVI_p90",
    "S2_NDVI_amplitude",
    "S2_valid_obs_count"
]

S2_SR = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CLOUDS = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

print("REFERENCE_CSV:", REFERENCE_CSV)
print("CHUNK_DIR:", CHUNK_DIR)
print("OUT_NPZ:", OUT_NPZ)
print("PATCH_SIZE:", PATCH_SIZE)
print("CLOUD_PROB_THRESHOLD:", CLOUD_PROB_THRESHOLD)
print("RECOVERY_CHUNK_SIZE:", RECOVERY_CHUNK_SIZE)


# ------------------------------------------------------------
# 2. Load reference table and current final cache
# ------------------------------------------------------------

ref_df = pd.read_csv(REFERENCE_CSV).copy()

if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(int).astype(str)
        + "_"
        + ref_df["observation_month"].astype(int).astype(str).str.zfill(2)
    )

ref_df["sample_id"] = ref_df["sample_id"].astype(str)
ref_df["unit_id"] = ref_df["unit_id"].astype(str)
ref_df["land_cover"] = ref_df["land_cover"].astype(str)
ref_df["observation_year"] = ref_df["observation_year"].astype(int)
ref_df["observation_month"] = ref_df["observation_month"].astype(int)
ref_df["latitude"] = ref_df["latitude"].astype(float)
ref_df["longitude"] = ref_df["longitude"].astype(float)
ref_df["row_index"] = np.arange(len(ref_df))

if not os.path.exists(OUT_NPZ):
    raise FileNotFoundError(f"Could not find existing final cache:\n{OUT_NPZ}")

current = np.load(OUT_NPZ, allow_pickle=True)
existing_sample_ids = set(current["sample_ids"].astype(str))

missing_df = ref_df[
    ~ref_df["sample_id"].isin(existing_sample_ids)
].copy()

print("Reference samples:", len(ref_df))
print("Samples currently in final cache:", len(existing_sample_ids))
print("Missing samples to recover:", len(missing_df))

display(
    missing_df[
        [
            "row_index",
            "sample_id",
            "unit_id",
            "observation_year",
            "observation_month",
            "land_cover",
            "latitude",
            "longitude"
        ]
    ].head(100)
)

if missing_df.empty:
    print("No missing samples. Nothing to recover.")


# ------------------------------------------------------------
# 3. Earth Engine helper functions
# ------------------------------------------------------------

def make_points_fc(df):
    features = []

    for _, row in df.iterrows():
        geom = ee.Geometry.Point([
            float(row["longitude"]),
            float(row["latitude"])
        ])

        props = {
            "sample_id": str(row["sample_id"]),
            "unit_id": str(row["unit_id"]),
            "land_cover": str(row["land_cover"]),
            "observation_year": int(row["observation_year"]),
            "observation_month": int(row["observation_month"]),
            "row_index": int(row["row_index"])
        }

        features.append(ee.Feature(geom, props))

    return ee.FeatureCollection(features)


def mask_s2_cloudprob_and_add_indices(joined_feature):
    joined_feature = ee.Feature(joined_feature)

    img = ee.Image(joined_feature.get("primary")).toFloat()
    cloud_prob = ee.Image(joined_feature.get("secondary"))

    scl = img.select("SCL")
    cloud_mask = cloud_prob.select("probability").lt(CLOUD_PROB_THRESHOLD)

    scl_mask = (
        scl.neq(3)    # cloud shadow
        .And(scl.neq(8))   # medium probability cloud
        .And(scl.neq(9))   # high probability cloud
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # snow/ice
    )

    img = img.updateMask(cloud_mask.And(scl_mask))

    blue = img.select("B2").divide(10000)
    green = img.select("B3").divide(10000)
    red = img.select("B4").divide(10000)
    nir = img.select("B8").divide(10000)
    swir1 = img.select("B11").divide(10000)
    swir2 = img.select("B12").divide(10000)

    eps = ee.Image.constant(1e-6)

    ndvi = nir.subtract(red).divide(nir.add(red).add(eps)).rename("NDVI")
    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(
            nir
            .add(red.multiply(6))
            .subtract(blue.multiply(7.5))
            .add(1)
        )
        .rename("EVI")
    )
    ndwi = green.subtract(nir).divide(green.add(nir).add(eps)).rename("NDWI")
    nbr = nir.subtract(swir2).divide(nir.add(swir2).add(eps)).rename("NBR")

    out = ee.Image.cat([
        blue.rename("B2"),
        green.rename("B3"),
        red.rename("B4"),
        nir.rename("B8"),
        swir1.rename("B11"),
        swir2.rename("B12"),
        ndvi,
        evi,
        ndwi,
        nbr
    ]).toFloat()

    return out.copyProperties(img, img.propertyNames())


def build_s2_annual_feature_image(year, region):
    year = int(year)

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    sr = (
        S2_SR
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
    )

    clouds = (
        S2_CLOUDS
        .filterBounds(region)
        .filterDate(start, end)
    )

    joined = ee.Join.inner().apply(
        primary=sr,
        secondary=clouds,
        condition=ee.Filter.equals(
            leftField="system:index",
            rightField="system:index"
        )
    )

    prepared = ee.ImageCollection(joined.map(mask_s2_cloudprob_and_add_indices))

    base_bands = [
        "B2", "B3", "B4", "B8", "B11", "B12",
        "NDVI", "EVI", "NDWI", "NBR"
    ]

    empty_base = (
        ee.Image.constant([0] * len(base_bands))
        .rename(base_bands)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )

    median_img = ee.Image(
        ee.Algorithms.If(
            prepared.size().gt(0),
            prepared.select(base_bands).median(),
            empty_base
        )
    )

    median_img = median_img.rename([
        "S2_B2_qm",
        "S2_B3_qm",
        "S2_B4_qm",
        "S2_B8_qm",
        "S2_B11_qm",
        "S2_B12_qm",
        "S2_NDVI_qm",
        "S2_EVI_qm",
        "S2_NDWI_qm",
        "S2_NBR_qm"
    ])

    ndvi_col = prepared.select("NDVI")

    ndvi_p10 = ee.Image(
        ee.Algorithms.If(
            prepared.size().gt(0),
            ndvi_col.reduce(ee.Reducer.percentile([10])).rename("S2_NDVI_p10"),
            ee.Image.constant(0).rename("S2_NDVI_p10").toFloat().updateMask(ee.Image.constant(0))
        )
    )

    ndvi_p90 = ee.Image(
        ee.Algorithms.If(
            prepared.size().gt(0),
            ndvi_col.reduce(ee.Reducer.percentile([90])).rename("S2_NDVI_p90"),
            ee.Image.constant(0).rename("S2_NDVI_p90").toFloat().updateMask(ee.Image.constant(0))
        )
    )

    ndvi_amp = ndvi_p90.subtract(ndvi_p10).rename("S2_NDVI_amplitude")

    valid_count = ee.Image(
        ee.Algorithms.If(
            prepared.size().gt(0),
            ndvi_col.count().rename("S2_valid_obs_count"),
            ee.Image.constant(0).rename("S2_valid_obs_count").toFloat()
        )
    )

    out = ee.Image.cat([
        median_img,
        ndvi_p10,
        ndvi_p90,
        ndvi_amp,
        valid_count
    ]).select(S2_FEATURE_NAMES).toFloat()

    return out


def get_s2_annual_patch_array_image(year, patch_size, region):
    img = build_s2_annual_feature_image(year, region)

    weights = [[1] * patch_size for _ in range(patch_size)]
    focus = patch_size // 2

    kernel = ee.Kernel.fixed(
        width=patch_size,
        height=patch_size,
        weights=weights,
        x=focus,
        y=focus,
        normalize=False
    )

    return img.neighborhoodToArray(
        kernel=kernel,
        defaultValue=0
    )


def extract_recovery_chunk(df_chunk, year):
    fc = make_points_fc(df_chunk)

    region = fc.geometry().bounds().buffer(1000)

    arr_img = get_s2_annual_patch_array_image(
        year=year,
        patch_size=PATCH_SIZE,
        region=region
    )

    sampled = arr_img.sampleRegions(
        collection=fc,
        properties=[
            "sample_id",
            "unit_id",
            "land_cover",
            "observation_year",
            "observation_month",
            "row_index"
        ],
        scale=10,
        geometries=False,
        tileScale=RECOVERY_TILE_SCALE
    )

    info = sampled.getInfo()

    X_list = []
    sample_ids = []
    unit_ids = []
    labels = []
    years = []
    months = []
    row_indices = []

    for f in info.get("features", []):
        props = f.get("properties", {})

        patch_channels = []
        ok = True

        for band in S2_FEATURE_NAMES:
            if band not in props:
                ok = False
                break

            arr = np.array(props[band], dtype=np.float32)

            if arr.shape != (PATCH_SIZE, PATCH_SIZE):
                ok = False
                break

            patch_channels.append(arr)

        if not ok:
            continue

        X_list.append(np.stack(patch_channels, axis=-1))
        sample_ids.append(str(props["sample_id"]))
        unit_ids.append(str(props["unit_id"]))
        labels.append(str(props["land_cover"]))
        years.append(int(props["observation_year"]))
        months.append(int(props["observation_month"]))
        row_indices.append(int(props["row_index"]))

    if len(X_list) == 0:
        return None

    X = np.stack(X_list).astype(np.float32)

    # Keep cache safe for model loading.
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    return {
        "X": X,
        "sample_ids": np.array(sample_ids).astype(str),
        "unit_ids": np.array(unit_ids).astype(str),
        "land_cover": np.array(labels).astype(str),
        "years": np.array(years).astype(np.int16),
        "months": np.array(months).astype(np.int8),
        "row_indices": np.array(row_indices).astype(np.int32),
        "channel_names": np.array(S2_FEATURE_NAMES).astype(str)
    }


# ------------------------------------------------------------
# 4. Recover missing samples only
# ------------------------------------------------------------

Path(CHUNK_DIR).mkdir(parents=True, exist_ok=True)

if not missing_df.empty:

    for year, g_year in missing_df.groupby("observation_year"):
        year = int(year)
        g_year = g_year.sort_values("row_index").reset_index(drop=True)

        print("\n" + "=" * 80)
        print(f"Recovering Sentinel-2 annual patches | year {year}")
        print(f"missing points: {len(g_year)}")
        print("=" * 80)

        for start in range(0, len(g_year), RECOVERY_CHUNK_SIZE):
            end = min(start + RECOVERY_CHUNK_SIZE, len(g_year))
            df_chunk = g_year.iloc[start:end].copy()

            row_min = int(df_chunk["row_index"].min())
            row_max = int(df_chunk["row_index"].max()) + 1

            recovery_path = os.path.join(
                CHUNK_DIR,
                (
                    f"s2_annual_cloudprob_p{PATCH_SIZE}_"
                    f"year{year}_recovery_rows{row_min}_{row_max}.npz"
                )
            )

            if os.path.exists(recovery_path):
                print("  exists, skip:", os.path.basename(recovery_path))
                continue

            print(
                f"  recovering rows {row_min}:{row_max} | "
                f"n={len(df_chunk)}"
            )

            try:
                result = extract_recovery_chunk(df_chunk, year)

                if result is None:
                    print("    WARNING: no valid samples returned.")
                    continue

                np.savez_compressed(
                    recovery_path,
                    X=result["X"],
                    sample_ids=result["sample_ids"],
                    unit_ids=result["unit_ids"],
                    land_cover=result["land_cover"],
                    years=result["years"],
                    months=result["months"],
                    row_indices=result["row_indices"],
                    channel_names=result["channel_names"]
                )

                print(
                    "    saved:",
                    os.path.basename(recovery_path),
                    "X:",
                    result["X"].shape
                )

            except Exception as e:
                print("    WARNING: recovery failed.")
                print("    rows:", row_min, row_max)
                print("    error:", repr(e))

            time.sleep(RECOVERY_SLEEP_SECONDS)


# ------------------------------------------------------------
# 5. Rebuild final NPZ from original + recovery chunks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("Rebuilding final cache from all chunk files")
print("=" * 80)

chunk_files = sorted([
    os.path.join(CHUNK_DIR, f)
    for f in os.listdir(CHUNK_DIR)
    if f.endswith(".npz")
])

print("Chunk files found:", len(chunk_files))

if len(chunk_files) == 0:
    raise ValueError("No chunk files found. Cannot rebuild final cache.")

X_parts = []
sample_parts = []
unit_parts = []
label_parts = []
year_parts = []
month_parts = []

for fp in tqdm(chunk_files, desc="Loading chunks"):
    d = np.load(fp, allow_pickle=True)

    X_parts.append(d["X"].astype(np.float32))
    sample_parts.append(d["sample_ids"].astype(str))
    unit_parts.append(d["unit_ids"].astype(str))
    label_parts.append(d["land_cover"].astype(str))

    # Support both possible internal key names.
    if "years" in d:
        year_parts.append(d["years"].astype(np.int16))
    else:
        year_parts.append(d["observation_year"].astype(np.int16))

    if "months" in d:
        month_parts.append(d["months"].astype(np.int8))
    else:
        month_parts.append(d["observation_month"].astype(np.int8))

X_all = np.concatenate(X_parts, axis=0).astype(np.float32)
sample_ids_all = np.concatenate(sample_parts).astype(str)
unit_ids_all = np.concatenate(unit_parts).astype(str)
land_cover_all = np.concatenate(label_parts).astype(str)
years_all = np.concatenate(year_parts).astype(np.int16)
months_all = np.concatenate(month_parts).astype(np.int8)

# Remove duplicate sample_ids, preferring the last occurrence.
# Because recovery files are alphabetically later than normal chunks,
# this safely keeps recovered versions if duplicates exist.
all_df = pd.DataFrame({
    "sample_id": sample_ids_all,
    "extract_idx": np.arange(len(sample_ids_all))
})

all_df = all_df.drop_duplicates(
    subset="sample_id",
    keep="last"
)

# Reorder exactly as reference CSV.
order_df = ref_df[[
    "sample_id",
    "unit_id",
    "land_cover",
    "observation_year",
    "observation_month",
    "latitude",
    "longitude"
]].copy()

if "zone_id" in ref_df.columns:
    order_df["zone_id"] = ref_df["zone_id"].astype(str)

order_df["order_idx"] = np.arange(len(order_df))

join_df = (
    order_df
    .merge(all_df, on="sample_id", how="inner")
    .sort_values("order_idx")
    .reset_index(drop=True)
)

idx = join_df["extract_idx"].to_numpy()

X_final = X_all[idx]
sample_ids_final = sample_ids_all[idx]
unit_ids_final = unit_ids_all[idx]
land_cover_final = land_cover_all[idx]
years_final = years_all[idx]
months_final = months_all[idx]

X_final = np.nan_to_num(
    X_final,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
).astype(np.float32)

missing_after = sorted(set(order_df["sample_id"]) - set(sample_ids_final))

print("Final X:", X_final.shape)
print("Final sample_ids:", sample_ids_final.shape)
print("Missing after recovery:", len(missing_after))

if len(missing_after) > 0:
    print("First missing after recovery:", missing_after[:50])

print("NaNs after cleanup:", int(np.isnan(X_final).sum()))
print("Infs after cleanup:", int(np.isinf(X_final).sum()))

# Save metadata next to NPZ.
metadata_path = OUT_NPZ.replace(".npz", "_metadata.csv")

metadata_df = join_df.drop(columns=["extract_idx"]).copy()
metadata_df.to_csv(metadata_path, index=False)

np.savez_compressed(
    OUT_NPZ,
    X=X_final,
    sample_ids=sample_ids_final,
    unit_ids=unit_ids_final,
    land_cover=land_cover_final,
    years=years_final,
    months=months_final,
    channel_names=np.array(S2_FEATURE_NAMES).astype(str),
    patch_size=np.array([PATCH_SIZE]),
    cloud_prob_threshold=np.array([CLOUD_PROB_THRESHOLD])
)

print("\nSaved repaired final cache:")
print(OUT_NPZ)

print("Saved metadata:")
print(metadata_path)

if len(missing_after) == 0:
    print("Done. All reference samples are now included.")
else:
    print("Done, but some samples still failed. Try RECOVERY_CHUNK_SIZE = 1 for the remaining ones.")

Earth Engine initialized.
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
CHUNK_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/chunks_cloudprob_p7
OUT_NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/Kasai_reference_s2_annual_cloudprob_median_amp_patch_p7_observation_year.npz
PATCH_SIZE: 7
CLOUD_PROB_THRESHOLD: 40
RECOVERY_CHUNK_SIZE: 1
Reference samples: 3706
Samples currently in final cache: 3701
Missing samples to recover: 5


,row_index,sample_id,unit_id,observation_year,observation_month,land_cover,latitude,longitude
2153,2153,KASAI_002154_2025_06,KASAI_002154,2025,6,water,-6.429509,23.803796
2154,2154,KASAI_002155_2025_06,KASAI_002155,2025,6,water,-6.423827,23.795536
2155,2155,KASAI_002156_2025_06,KASAI_002156,2025,6,water,-6.417737,23.796873
2156,2156,KASAI_002157_2025_06,KASAI_002157,2025,6,water,-6.415059,23.799043
3296,3296,KASAI_003297_2025_04,KASAI_003297,2025,4,water,-3.031535,16.881456



Recovering Sentinel-2 annual patches | year 2025
missing points: 5
  recovering rows 2153:2154 | n=1
    saved: s2_annual_cloudprob_p7_year2025_recovery_rows2153_2154.npz X: (1, 7, 7, 14)
  recovering rows 2154:2155 | n=1
    saved: s2_annual_cloudprob_p7_year2025_recovery_rows2154_2155.npz X: (1, 7, 7, 14)
  recovering rows 2155:2156 | n=1
    saved: s2_annual_cloudprob_p7_year2025_recovery_rows2155_2156.npz X: (1, 7, 7, 14)
  recovering rows 2156:2157 | n=1
    saved: s2_annual_cloudprob_p7_year2025_recovery_rows2156_2157.npz X: (1, 7, 7, 14)
  recovering rows 3296:3297 | n=1
    saved: s2_annual_cloudprob_p7_year2025_recovery_rows3296_3297.npz X: (1, 7, 7, 14)

Rebuilding final cache from all chunk files
Chunk files found: 114


Loading chunks:   0%|          | 0/114 [00:00<?, ?it/s]

Final X: (3706, 7, 7, 14)
Final sample_ids: (3706,)
Missing after recovery: 0
NaNs after cleanup: 0
Infs after cleanup: 0

Saved repaired final cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/Kasai_reference_s2_annual_cloudprob_median_amp_patch_p7_observation_year.npz
Saved metadata:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_annual_patches/Kasai_reference_s2_annual_cloudprob_median_amp_patch_p7_observation_year_metadata.csv
Done. All reference samples are now included.


# SEN2 Texture patch

In [ ]:
# ============================================================
# CACHE BLOCK — Sentinel-2 NDVI texture patch cache
#
# Memory-safe version:
#   - Uses S2 SR Harmonized + S2 Cloud Probability
#   - Uses SCL + cloud probability masking
#   - Builds annual NDVI quality mosaic per year/chunk region
#   - Computes simple NDVI texture metrics:
#       NDVI
#       NDVI focal mean
#       NDVI focal stdDev
#       NDVI focal min
#       NDVI focal max
#       NDVI focal range
#   - Extracts texture patches around observation centroids
#   - Saves each year/chunk separately
#   - Recombines all chunks into one final NPZ
#
# Output:
#   NN/Data/Cache/Sentinel2_texture_patches/
#     chunks_p7/
#       s2_texture_ndvi_p7_year2020_chunk0000.npz
#       ...
#     Kasai_reference_s2_texture_ndvi_patch_p7_observation_year.npz
# ============================================================


# ------------------------------------------------------------
# 1. Earth Engine auth/init
# ------------------------------------------------------------

import ee

try:
    ee.Initialize(project="panafricanlu")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="panafricanlu")

print("Earth Engine initialized.")


# ------------------------------------------------------------
# 2. Imports and paths
# ------------------------------------------------------------

import os
import time
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN"

REFERENCE_CSV = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Kasai_reference.csv"
)

S2_TEXTURE_PATCH_DIR = os.path.join(
    PROJECT_ROOT,
    "Data",
    "Cache",
    "Sentinel2_texture_patches"
)

PATCH_SIZE = 7
TEXTURE_KERNEL_RADIUS_PIXELS = 1   # radius 1 = 3x3 texture window
CHUNK_SIZE = 30                    # keep small; use 20 if memory errors persist
S2_SCALE = 10
FILL_VALUE = -9999.0

CLOUD_PROB_THRESHOLD = 40
MAX_SCENE_CLOUD_PERCENT = 85

OVERWRITE_CHUNKS = False
OVERWRITE_FINAL = True

CHUNK_DIR = os.path.join(
    S2_TEXTURE_PATCH_DIR,
    f"chunks_p{PATCH_SIZE}"
)

os.makedirs(S2_TEXTURE_PATCH_DIR, exist_ok=True)
os.makedirs(CHUNK_DIR, exist_ok=True)

OUT_NPZ = os.path.join(
    S2_TEXTURE_PATCH_DIR,
    f"Kasai_reference_s2_texture_ndvi_patch_p{PATCH_SIZE}_observation_year.npz"
)

print("REFERENCE_CSV:", REFERENCE_CSV)
print("S2_TEXTURE_PATCH_DIR:", S2_TEXTURE_PATCH_DIR)
print("CHUNK_DIR:", CHUNK_DIR)
print("OUT_NPZ:", OUT_NPZ)
print("PATCH_SIZE:", PATCH_SIZE)
print("TEXTURE_KERNEL_RADIUS_PIXELS:", TEXTURE_KERNEL_RADIUS_PIXELS)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("CLOUD_PROB_THRESHOLD:", CLOUD_PROB_THRESHOLD)


# ------------------------------------------------------------
# 3. Load reference CSV
# ------------------------------------------------------------

ref_df = pd.read_csv(REFERENCE_CSV)

required_cols = [
    "unit_id",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "zone_id",
    "land_cover"
]

missing = [c for c in required_cols if c not in ref_df.columns]
if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))

ref_df["unit_id"] = ref_df["unit_id"].astype(str)
ref_df["observation_year"] = ref_df["observation_year"].astype(int)
ref_df["observation_month"] = ref_df["observation_month"].astype(int)

if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(str)
        + "_"
        + ref_df["observation_month"].astype(str).str.zfill(2)
    )

ref_df["sample_id"] = ref_df["sample_id"].astype(str)

ref_df = (
    ref_df
    .dropna(subset=["latitude", "longitude"])
    .drop_duplicates(subset=["sample_id"], keep="first")
    .reset_index(drop=True)
)

ref_df["row_index"] = np.arange(len(ref_df))

print("Reference rows:", len(ref_df))
print("Years:", sorted(ref_df["observation_year"].unique()))
display(ref_df.head())
display(ref_df["land_cover"].value_counts().reset_index(name="n"))


# ------------------------------------------------------------
# 4. Sentinel-2 collections and texture channels
# ------------------------------------------------------------

s2_sr = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
s2_clouds = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

S2_TEXTURE_CHANNELS = [
    "S2_NDVI_qm",
    "S2_NDVI_mean_3x3",
    "S2_NDVI_std_3x3",
    "S2_NDVI_min_3x3",
    "S2_NDVI_max_3x3",
    "S2_NDVI_range_3x3"
]


# ------------------------------------------------------------
# 5. Sentinel-2 cloud masking and NDVI preparation
# ------------------------------------------------------------

def empty_texture_image():
    return (
        ee.Image.constant([FILL_VALUE] * len(S2_TEXTURE_CHANNELS))
        .rename(S2_TEXTURE_CHANNELS)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )


def join_s2_with_cloud_probability(sr_col, cloud_col):
    joined = ee.Join.saveFirst("cloud_prob_img").apply(
        primary=sr_col,
        secondary=cloud_col,
        condition=ee.Filter.equals(
            leftField="system:index",
            rightField="system:index"
        )
    )
    return ee.ImageCollection(joined)


def mask_s2_cloudprob_and_scl(img):
    cloud_prob_img = ee.Image(img.get("cloud_prob_img"))
    cloud_prob = cloud_prob_img.select("probability")

    scl = img.select("SCL")

    cloud_prob_mask = cloud_prob.lt(CLOUD_PROB_THRESHOLD)

    scl_mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    clear_score = ee.Image.constant(100).subtract(cloud_prob).rename("clear_score")

    masked = img.updateMask(cloud_prob_mask.And(scl_mask))

    return masked.addBands(clear_score).copyProperties(img, img.propertyNames())


def add_ndvi_and_clear_score(img):
    red = img.select("B4").toFloat().divide(10000)
    nir = img.select("B8").toFloat().divide(10000)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    clear_score = img.select("clear_score")

    return ee.Image.cat([ndvi, clear_score]).toFloat().copyProperties(img, img.propertyNames())


def get_s2_ndvi_collection_for_year(year, region):
    start = ee.Date.fromYMD(int(year), 1, 1)
    end = start.advance(1, "year")

    sr_col = (
        s2_sr
        .filterDate(start, end)
        .filterBounds(region)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PERCENT))
    )

    cloud_col = (
        s2_clouds
        .filterDate(start, end)
        .filterBounds(region)
    )

    joined = join_s2_with_cloud_probability(sr_col, cloud_col)

    prepared = (
        joined
        .filter(ee.Filter.notNull(["cloud_prob_img"]))
        .map(mask_s2_cloudprob_and_scl)
        .map(add_ndvi_and_clear_score)
    )

    return prepared


# ------------------------------------------------------------
# 6. Build annual NDVI texture image
# ------------------------------------------------------------

def build_s2_ndvi_texture_image(year, region):
    col = get_s2_ndvi_collection_for_year(year, region)

    def make_image():
        ndvi_qm = (
            col
            .select(["NDVI", "clear_score"])
            .qualityMosaic("clear_score")
            .select("NDVI")
            .rename("S2_NDVI_qm")
            .toFloat()
        )

        texture_kernel = ee.Kernel.square(
            radius=TEXTURE_KERNEL_RADIUS_PIXELS,
            units="pixels",
            normalize=False
        )

        ndvi_mean = (
            ndvi_qm
            .reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=texture_kernel
            )
            .rename("S2_NDVI_mean_3x3")
            .toFloat()
        )

        ndvi_std = (
            ndvi_qm
            .reduceNeighborhood(
                reducer=ee.Reducer.stdDev(),
                kernel=texture_kernel
            )
            .rename("S2_NDVI_std_3x3")
            .toFloat()
        )

        ndvi_min = (
            ndvi_qm
            .reduceNeighborhood(
                reducer=ee.Reducer.min(),
                kernel=texture_kernel
            )
            .rename("S2_NDVI_min_3x3")
            .toFloat()
        )

        ndvi_max = (
            ndvi_qm
            .reduceNeighborhood(
                reducer=ee.Reducer.max(),
                kernel=texture_kernel
            )
            .rename("S2_NDVI_max_3x3")
            .toFloat()
        )

        ndvi_range = (
            ndvi_max
            .subtract(ndvi_min)
            .rename("S2_NDVI_range_3x3")
            .toFloat()
        )

        return ee.Image.cat([
            ndvi_qm,
            ndvi_mean,
            ndvi_std,
            ndvi_min,
            ndvi_max,
            ndvi_range
        ]).select(S2_TEXTURE_CHANNELS).toFloat()

    img = ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            make_image(),
            empty_texture_image()
        )
    )

    return img.select(S2_TEXTURE_CHANNELS).toFloat()


def image_to_patch_array(img, patch_size):
    weights = [[1] * patch_size for _ in range(patch_size)]
    focus = patch_size // 2

    kernel = ee.Kernel.fixed(
        width=patch_size,
        height=patch_size,
        weights=weights,
        x=focus,
        y=focus,
        normalize=False
    )

    return img.neighborhoodToArray(
        kernel=kernel,
        defaultValue=FILL_VALUE
    )


# ------------------------------------------------------------
# 7. FeatureCollection helpers
# ------------------------------------------------------------

def chunk_to_fc(df):
    features = []

    for _, row in df.iterrows():
        lon = float(row["longitude"])
        lat = float(row["latitude"])

        props = {
            "sample_id": str(row["sample_id"]),
            "unit_id": str(row["unit_id"]),
            "zone_id": str(row["zone_id"]),
            "land_cover": str(row["land_cover"]),
            "observation_year": int(row["observation_year"]),
            "observation_month": int(row["observation_month"]),
            "row_index": int(row["row_index"])
        }

        features.append(
            ee.Feature(
                ee.Geometry.Point([lon, lat]),
                props
            )
        )

    return ee.FeatureCollection(features)


def chunk_region(df, buffer_m=1200):
    fc = chunk_to_fc(df)
    return fc.geometry().buffer(buffer_m).bounds()


# ------------------------------------------------------------
# 8. Extract one year/chunk
# ------------------------------------------------------------

def extract_one_texture_chunk(year, chunk_df, chunk_id):
    chunk_path = os.path.join(
        CHUNK_DIR,
        f"s2_texture_ndvi_p{PATCH_SIZE}_year{int(year)}_chunk{chunk_id:04d}.npz"
    )

    if os.path.exists(chunk_path) and not OVERWRITE_CHUNKS:
        print("    exists, skip:", os.path.basename(chunk_path))
        return chunk_path

    region = chunk_region(chunk_df, buffer_m=1200)

    texture_img = build_s2_ndvi_texture_image(year, region)
    arr_img = image_to_patch_array(texture_img, PATCH_SIZE)

    fc = chunk_to_fc(chunk_df)

    sampled = arr_img.sampleRegions(
        collection=fc,
        properties=[
            "sample_id",
            "unit_id",
            "zone_id",
            "land_cover",
            "observation_year",
            "observation_month",
            "row_index"
        ],
        scale=S2_SCALE,
        geometries=False,
        tileScale=8
    )

    info = sampled.getInfo()

    X_list = []
    sample_id_list = []
    unit_id_list = []
    zone_list = []
    label_list = []
    year_list = []
    month_list = []
    row_index_list = []

    for feat in info["features"]:
        props = feat["properties"]

        patch_channels = []
        ok = True

        for band in S2_TEXTURE_CHANNELS:
            if band not in props:
                ok = False
                break

            arr = np.array(props[band], dtype=np.float32)

            if arr.shape != (PATCH_SIZE, PATCH_SIZE):
                ok = False
                break

            arr[arr == FILL_VALUE] = np.nan
            patch_channels.append(arr)

        if not ok:
            continue

        X_list.append(np.stack(patch_channels, axis=-1))
        sample_id_list.append(str(props["sample_id"]))
        unit_id_list.append(str(props["unit_id"]))
        zone_list.append(str(props["zone_id"]))
        label_list.append(str(props["land_cover"]))
        year_list.append(int(props["observation_year"]))
        month_list.append(int(props["observation_month"]))
        row_index_list.append(int(props["row_index"]))

    if len(X_list) == 0:
        print("    WARNING: no texture patches returned.")
        return None

    X_chunk = np.stack(X_list).astype(np.float32)

    np.savez_compressed(
        chunk_path,
        X=X_chunk,
        sample_ids=np.array(sample_id_list).astype(str),
        unit_ids=np.array(unit_id_list).astype(str),
        zone_id=np.array(zone_list).astype(str),
        land_cover=np.array(label_list).astype(str),
        observation_year=np.array(year_list).astype(int),
        observation_month=np.array(month_list).astype(int),
        row_index=np.array(row_index_list).astype(int),
        channel_names=np.array(S2_TEXTURE_CHANNELS).astype(str),
        patch_size=np.array([PATCH_SIZE]).astype(int),
        texture_kernel_radius_pixels=np.array([TEXTURE_KERNEL_RADIUS_PIXELS]).astype(int),
        cloud_prob_threshold=np.array([CLOUD_PROB_THRESHOLD]).astype(int)
    )

    print("    saved:", os.path.basename(chunk_path), "X:", X_chunk.shape)

    return chunk_path


# ------------------------------------------------------------
# 9. Extract all chunks, resumable
# ------------------------------------------------------------

if os.path.exists(OUT_NPZ) and not OVERWRITE_FINAL:
    print("Final texture cache already exists:")
    print(OUT_NPZ)
    print("Set OVERWRITE_FINAL=True if you want to rebuild final file.")

else:
    years = sorted(ref_df["observation_year"].unique())

    for year in years:
        year_df = ref_df[ref_df["observation_year"] == year].copy()
        year_df = year_df.reset_index(drop=True)

        n_chunks = int(np.ceil(len(year_df) / CHUNK_SIZE))

        print("\n" + "=" * 80)
        print(f"Extracting Sentinel-2 NDVI texture patches | year {year}")
        print(f"points: {len(year_df)} | chunks: {n_chunks}")
        print("=" * 80)

        for chunk_id, start in enumerate(range(0, len(year_df), CHUNK_SIZE)):
            end = min(start + CHUNK_SIZE, len(year_df))
            chunk_df = year_df.iloc[start:end].copy()

            print(f"  chunk {chunk_id + 1}/{n_chunks} | rows {start}:{end}")

            try:
                extract_one_texture_chunk(year, chunk_df, chunk_id)

            except Exception as e:
                print("    WARNING: texture chunk failed.")
                print("    year:", year)
                print("    chunk:", chunk_id)
                print("    rows:", start, end)
                print("    error:", repr(e))
                print("    Continuing with next chunk.")

            time.sleep(0.5)

    print("\nTexture extraction pass finished.")


# ------------------------------------------------------------
# 10. Combine chunk files into final NPZ
# ------------------------------------------------------------

chunk_files = sorted(glob.glob(
    os.path.join(CHUNK_DIR, f"s2_texture_ndvi_p{PATCH_SIZE}_year*_chunk*.npz")
))

if len(chunk_files) == 0:
    raise ValueError("No texture chunk files found. Extraction did not produce cache chunks.")

print("Texture chunk files found:", len(chunk_files))

X_all = []
sample_ids_all = []
unit_ids_all = []
zone_all = []
label_all = []
year_all = []
month_all = []
row_index_all = []

for path in tqdm(chunk_files, desc="Combining texture chunks"):
    data = np.load(path, allow_pickle=True)

    X_all.append(data["X"].astype(np.float32))
    sample_ids_all.append(data["sample_ids"].astype(str))
    unit_ids_all.append(data["unit_ids"].astype(str))
    zone_all.append(data["zone_id"].astype(str))
    label_all.append(data["land_cover"].astype(str))
    year_all.append(data["observation_year"].astype(int))
    month_all.append(data["observation_month"].astype(int))
    row_index_all.append(data["row_index"].astype(int))

X_all = np.concatenate(X_all, axis=0)
sample_ids_all = np.concatenate(sample_ids_all)
unit_ids_all = np.concatenate(unit_ids_all)
zone_all = np.concatenate(zone_all)
label_all = np.concatenate(label_all)
year_all = np.concatenate(year_all)
month_all = np.concatenate(month_all)
row_index_all = np.concatenate(row_index_all)

# Sort back to original reference CSV order.
order = np.argsort(row_index_all)

X_all = X_all[order]
sample_ids_all = sample_ids_all[order]
unit_ids_all = unit_ids_all[order]
zone_all = zone_all[order]
label_all = label_all[order]
year_all = year_all[order]
month_all = month_all[order]
row_index_all = row_index_all[order]

# Drop duplicated sample IDs if any.
seen = set()
keep_idx = []

for i, sid in enumerate(sample_ids_all):
    if sid not in seen:
        seen.add(sid)
        keep_idx.append(i)

keep_idx = np.array(keep_idx)

X_all = X_all[keep_idx]
sample_ids_all = sample_ids_all[keep_idx]
unit_ids_all = unit_ids_all[keep_idx]
zone_all = zone_all[keep_idx]
label_all = label_all[keep_idx]
year_all = year_all[keep_idx]
month_all = month_all[keep_idx]
row_index_all = row_index_all[keep_idx]

np.savez_compressed(
    OUT_NPZ,
    X=X_all.astype(np.float32),
    sample_ids=sample_ids_all.astype(str),
    unit_ids=unit_ids_all.astype(str),
    zone_id=zone_all.astype(str),
    land_cover=label_all.astype(str),
    observation_year=year_all.astype(int),
    observation_month=month_all.astype(int),
    row_index=row_index_all.astype(int),
    channel_names=np.array(S2_TEXTURE_CHANNELS).astype(str),
    patch_size=np.array([PATCH_SIZE]).astype(int),
    texture_kernel_radius_pixels=np.array([TEXTURE_KERNEL_RADIUS_PIXELS]).astype(int),
    cloud_prob_threshold=np.array([CLOUD_PROB_THRESHOLD]).astype(int)
)

print("\nSaved final Sentinel-2 NDVI texture patch cache:")
print(OUT_NPZ)
print("X:", X_all.shape)
print("sample_ids:", sample_ids_all.shape)
print("channels:", len(S2_TEXTURE_CHANNELS), S2_TEXTURE_CHANNELS)

expected = set(ref_df["sample_id"].astype(str))
got = set(sample_ids_all.astype(str))
missing = sorted(expected - got)

print("\nMissing samples compared with reference:")
print("Missing:", len(missing))
print("First missing:", missing[:20])

print("\nNaN diagnostics:")
print("Total NaNs:", int(np.isnan(X_all).sum()))
print("NaNs per channel:")
nan_per_channel = np.isnan(X_all).sum(axis=(0, 1, 2))
for name, n in zip(S2_TEXTURE_CHANNELS, nan_per_channel):
    print(f"  {name}: {int(n)}")

print("Done.")

Earth Engine initialized.
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
S2_TEXTURE_PATCH_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_texture_patches
CHUNK_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_texture_patches/chunks_p7
OUT_NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_texture_patches/Kasai_reference_s2_texture_ndvi_patch_p7_observation_year.npz
PATCH_SIZE: 7
TEXTURE_KERNEL_RADIUS_PIXELS: 1
CHUNK_SIZE: 30
CLOUD_PROB_THRESHOLD: 40
Reference rows: 3706
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id,row_index
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020_04,0
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020_04,1
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020_04,2
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020_04,3
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020_04,4


,land_cover,n
0,cropland,1039
1,savannah,741
2,water,738
3,forest,598
4,built_up,590



Extracting Sentinel-2 NDVI texture patches | year 2020
points: 1180 | chunks: 40
  chunk 1/40 | rows 0:30
    saved: s2_texture_ndvi_p7_year2020_chunk0000.npz X: (30, 7, 7, 6)
  chunk 2/40 | rows 30:60
    saved: s2_texture_ndvi_p7_year2020_chunk0001.npz X: (30, 7, 7, 6)
  chunk 3/40 | rows 60:90
    saved: s2_texture_ndvi_p7_year2020_chunk0002.npz X: (30, 7, 7, 6)
  chunk 4/40 | rows 90:120
    saved: s2_texture_ndvi_p7_year2020_chunk0003.npz X: (30, 7, 7, 6)
  chunk 5/40 | rows 120:150
    saved: s2_texture_ndvi_p7_year2020_chunk0004.npz X: (30, 7, 7, 6)
  chunk 6/40 | rows 150:180
    saved: s2_texture_ndvi_p7_year2020_chunk0005.npz X: (30, 7, 7, 6)
  chunk 7/40 | rows 180:210
    saved: s2_texture_ndvi_p7_year2020_chunk0006.npz X: (30, 7, 7, 6)
  chunk 8/40 | rows 210:240
    saved: s2_texture_ndvi_p7_year2020_chunk0007.npz X: (30, 7, 7, 6)
  chunk 9/40 | rows 240:270
    saved: s2_texture_ndvi_p7_year2020_chunk0008.npz X: (30, 7, 7, 6)
  chunk 10/40 | rows 270:300
    saved: s2_t

    saved: s2_texture_ndvi_p7_year2025_chunk0008.npz X: (30, 7, 7, 6)
  chunk 10/11 | rows 270:300
    saved: s2_texture_ndvi_p7_year2025_chunk0009.npz X: (30, 7, 7, 6)
  chunk 11/11 | rows 300:324
    saved: s2_texture_ndvi_p7_year2025_chunk0010.npz X: (24, 7, 7, 6)

Texture extraction pass finished.
Texture chunk files found: 127


Combining texture chunks:   0%|          | 0/127 [00:00<?, ?it/s]


Saved final Sentinel-2 NDVI texture patch cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/Sentinel2_texture_patches/Kasai_reference_s2_texture_ndvi_patch_p7_observation_year.npz
X: (3706, 7, 7, 6)
sample_ids: (3706,)
channels: 6 ['S2_NDVI_qm', 'S2_NDVI_mean_3x3', 'S2_NDVI_std_3x3', 'S2_NDVI_min_3x3', 'S2_NDVI_max_3x3', 'S2_NDVI_range_3x3']

Missing samples compared with reference:
Missing: 0
First missing: []

NaN diagnostics:
Total NaNs: 6
NaNs per channel:
  S2_NDVI_qm: 1
  S2_NDVI_mean_3x3: 1
  S2_NDVI_std_3x3: 1
  S2_NDVI_min_3x3: 1
  S2_NDVI_max_3x3: 1
  S2_NDVI_range_3x3: 1
Done.


# NICFI patches

In [ ]:
# ============================================================
# NICFI biannual/semester relative + texture patch cache
# ============================================================
#
# Purpose:
#   Build high-resolution NICFI patch caches using only relative
#   spectral metrics and local texture metrics, avoiding unstable
#   raw NICFI B/G/R/N values.
#
# Important revision:
#   NICFI semester image selection now uses:
#     1. monthly NICFI basemaps aggregated to H1/H2
#     2. biannual NICFI fallback only if monthly is unavailable
#
# Output:
#   Data/Cache/nicfi_biannual_relative_texture_patches/
#
# Final NPZ name:
#   Kasai_reference_nicfi_biannual_relative_texture_patch_p15_observation_year.npz
# ============================================================


# ------------------------------------------------------------
# 1. Imports and Earth Engine initialization
# ------------------------------------------------------------

import os
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import ee

try:
    ee.Initialize(project="panafricanlu")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="panafricanlu")

print("Earth Engine initialized.")


# ------------------------------------------------------------
# 2. Paths and settings
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN"
DATA_DIR = os.path.join(PROJECT_ROOT, "Data")

REFERENCE_CSV = os.path.join(DATA_DIR, "Kasai_reference.csv")

CACHE_DIR = os.path.join(
    DATA_DIR,
    "Cache",
    "nicfi_biannual_relative_texture_patches"
)

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

PATCH_SIZES = [9, 15, 21]

CHUNK_SIZE = 40
PLANET_SCALE = 4.77

TEXTURE_RADIUS_PIXELS = 1

# Keep chunks reusable.
# Set True only if you want to rerun already saved chunks.
OVERWRITE_CHUNKS = False

# Important: previous final cache was incomplete, so overwrite final assembly.
OVERWRITE_FINAL = True

PLANET_IC = "projects/planet-nicfi/assets/basemaps/africa"

print("REFERENCE_CSV:", REFERENCE_CSV)
print("CACHE_DIR:", CACHE_DIR)
print("PATCH_SIZES:", PATCH_SIZES)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("PLANET_SCALE:", PLANET_SCALE)


# ------------------------------------------------------------
# 3. Read reference table
# ------------------------------------------------------------

if not os.path.exists(REFERENCE_CSV):
    raise FileNotFoundError(f"Could not find reference CSV:\n{REFERENCE_CSV}")

ref_df = pd.read_csv(REFERENCE_CSV)

required_cols = [
    "unit_id",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "land_cover"
]

missing = [c for c in required_cols if c not in ref_df.columns]
if missing:
    raise ValueError("Missing required columns in reference CSV: " + ", ".join(missing))

ref_df = ref_df.copy()

ref_df["unit_id"] = ref_df["unit_id"].astype(str)
ref_df["observation_year"] = ref_df["observation_year"].astype(int)
ref_df["observation_month"] = ref_df["observation_month"].astype(int)
ref_df["land_cover"] = ref_df["land_cover"].astype(str)

if "sample_id" not in ref_df.columns:
    ref_df["sample_id"] = (
        ref_df["unit_id"].astype(str)
        + "_"
        + ref_df["observation_year"].astype(str)
        + "_"
        + ref_df["observation_month"].astype(str).str.zfill(2)
    )
else:
    ref_df["sample_id"] = ref_df["sample_id"].astype(str)

ref_df["nicfi_semester"] = np.where(
    ref_df["observation_month"] <= 6,
    "H1",
    "H2"
)

n_before = len(ref_df)
ref_df = ref_df[ref_df["observation_year"] >= 2015].copy()
n_after = len(ref_df)

print("Reference rows before year filter:", n_before)
print("Reference rows after year >= 2015:", n_after)

n_before_dups = len(ref_df)
ref_df = ref_df.drop_duplicates(subset=["sample_id"], keep="first").reset_index(drop=True)
n_after_dups = len(ref_df)

print("Rows before duplicate sample_id removal:", n_before_dups)
print("Rows after duplicate sample_id removal:", n_after_dups)

display(ref_df.head())
display(ref_df["land_cover"].value_counts().reset_index(name="n"))


# ------------------------------------------------------------
# 4. NICFI relative + texture feature definitions
# ------------------------------------------------------------

nicfi_all = ee.ImageCollection(PLANET_IC)

PLANET_BANDS = ["B", "G", "R", "N"]

RELATIVE_FEATURES = [
    "NDVI",
    "GNDVI",
    "NDWI",
    "BRIGHTNESS",
    "RED_FRAC",
    "GREEN_FRAC",
    "BLUE_FRAC",
    "NIR_FRAC"
]

TEXTURE_SOURCE_BANDS = [
    "NDVI",
    "BRIGHTNESS",
    "RED_FRAC",
    "GREEN_FRAC",
    "NIR_FRAC"
]

TEXTURE_SUFFIXES = [
    "MEAN",
    "STD",
    "RANGE",
    "CENTER_MINUS_MEAN"
]

TEXTURE_FEATURES = [
    f"{b}_{s}"
    for b in TEXTURE_SOURCE_BANDS
    for s in TEXTURE_SUFFIXES
]

NICFI_FEATURES = RELATIVE_FEATURES + TEXTURE_FEATURES

print("Number of NICFI derived features:", len(NICFI_FEATURES))
print(NICFI_FEATURES)


def empty_raw_nicfi_image():
    return (
        ee.Image.constant([0, 0, 0, 0])
        .rename(PLANET_BANDS)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )


def empty_nicfi_feature_image():
    return (
        ee.Image.constant([0] * len(NICFI_FEATURES))
        .rename(NICFI_FEATURES)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )


def get_semester_dates(year, semester):
    year = int(year)

    if semester == "H1":
        start = ee.Date.fromYMD(year, 1, 1)
        end = ee.Date.fromYMD(year, 7, 1)
    elif semester == "H2":
        start = ee.Date.fromYMD(year, 7, 1)
        end = ee.Date.fromYMD(year + 1, 1, 1)
    else:
        raise ValueError("semester must be 'H1' or 'H2'")

    return start, end


def add_nicfi_relative_features(img):
    """
    Convert raw NICFI B/G/R/N into relative spectral features.
    Avoids feeding unstable raw band values to the model.
    """
    img = img.select(PLANET_BANDS).toFloat()

    B = img.select("B")
    G = img.select("G")
    R = img.select("R")
    N = img.select("N")

    eps = ee.Image.constant(1e-6)

    vis_sum = B.add(G).add(R).add(eps)
    all_sum = B.add(G).add(R).add(N).add(eps)

    ndvi = N.subtract(R).divide(N.add(R).add(eps)).rename("NDVI")
    gndvi = N.subtract(G).divide(N.add(G).add(eps)).rename("GNDVI")
    ndwi = G.subtract(N).divide(G.add(N).add(eps)).rename("NDWI")

    brightness = B.add(G).add(R).divide(3).rename("BRIGHTNESS")

    red_frac = R.divide(vis_sum).rename("RED_FRAC")
    green_frac = G.divide(vis_sum).rename("GREEN_FRAC")
    blue_frac = B.divide(vis_sum).rename("BLUE_FRAC")
    nir_frac = N.divide(all_sum).rename("NIR_FRAC")

    return ee.Image.cat([
        ndvi,
        gndvi,
        ndwi,
        brightness,
        red_frac,
        green_frac,
        blue_frac,
        nir_frac
    ]).toFloat()


def add_local_texture(img, radius_pixels=1):
    """
    Add local texture features to selected relative bands.
    """
    kernel = ee.Kernel.square(
        radius=radius_pixels,
        units="pixels",
        normalize=False
    )

    texture_layers = []

    for band in TEXTURE_SOURCE_BANDS:
        x = img.select(band)

        local_mean = x.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel
        ).rename(band + "_MEAN")

        local_std = x.reduceNeighborhood(
            reducer=ee.Reducer.stdDev(),
            kernel=kernel
        ).rename(band + "_STD")

        local_min = x.reduceNeighborhood(
            reducer=ee.Reducer.min(),
            kernel=kernel
        )

        local_max = x.reduceNeighborhood(
            reducer=ee.Reducer.max(),
            kernel=kernel
        )

        local_range = local_max.subtract(local_min).rename(band + "_RANGE")

        center_minus_mean = x.subtract(local_mean).rename(
            band + "_CENTER_MINUS_MEAN"
        )

        texture_layers.extend([
            local_mean,
            local_std,
            local_range,
            center_minus_mean
        ])

    return img.addBands(texture_layers).select(NICFI_FEATURES).toFloat()


def get_nicfi_semester_image(year, semester, region=None):
    """
    Build a NICFI semester image.

    Priority:
      1. Monthly NICFI basemaps aggregated over the semester.
      2. Biannual NICFI basemaps in the same semester.
      3. Empty masked image if neither exists.

    This fixes the issue where only 2020 H1 biannual imagery was found.
    """
    start, end = get_semester_dates(year, semester)

    monthly_col = (
        nicfi_all
        .filter(ee.Filter.eq("cadence", "monthly"))
        .filterDate(start, end)
    )

    biannual_col = (
        nicfi_all
        .filter(ee.Filter.eq("cadence", "biannual"))
        .filterDate(start, end)
    )

    if region is not None:
        monthly_col = monthly_col.filterBounds(region)
        biannual_col = biannual_col.filterBounds(region)

    monthly_raw = ee.Image(
        ee.Algorithms.If(
            monthly_col.size().gt(0),
            monthly_col.median().select(PLANET_BANDS).toFloat(),
            empty_raw_nicfi_image()
        )
    )

    biannual_raw = ee.Image(
        ee.Algorithms.If(
            biannual_col.size().gt(0),
            biannual_col.median().select(PLANET_BANDS).toFloat(),
            empty_raw_nicfi_image()
        )
    )

    raw_img = ee.Image(
        ee.Algorithms.If(
            monthly_col.size().gt(0),
            monthly_raw,
            ee.Algorithms.If(
                biannual_col.size().gt(0),
                biannual_raw,
                empty_raw_nicfi_image()
            )
        )
    )

    rel = add_nicfi_relative_features(raw_img)
    feat = add_local_texture(rel, radius_pixels=TEXTURE_RADIUS_PIXELS)

    return feat.select(NICFI_FEATURES).toFloat()


def get_patch_array_image(year, semester, patch_size, region=None):
    img = get_nicfi_semester_image(
        year=year,
        semester=semester,
        region=region
    )

    weights = [[1] * patch_size for _ in range(patch_size)]
    focus = patch_size // 2

    kernel = ee.Kernel.fixed(
        width=patch_size,
        height=patch_size,
        weights=weights,
        x=focus,
        y=focus,
        normalize=False
    )

    return img.neighborhoodToArray(
        kernel=kernel,
        defaultValue=0
    )


# ------------------------------------------------------------
# 4b. Diagnostic: NICFI availability by semester
# ------------------------------------------------------------

diag_rows = []

for year in sorted(ref_df["observation_year"].unique()):
    for semester in ["H1", "H2"]:
        start, end = get_semester_dates(int(year), semester)

        monthly_count = (
            nicfi_all
            .filter(ee.Filter.eq("cadence", "monthly"))
            .filterDate(start, end)
            .size()
            .getInfo()
        )

        biannual_count = (
            nicfi_all
            .filter(ee.Filter.eq("cadence", "biannual"))
            .filterDate(start, end)
            .size()
            .getInfo()
        )

        diag_rows.append({
            "year": int(year),
            "semester": semester,
            "monthly_count": monthly_count,
            "biannual_count": biannual_count
        })

nicfi_diag_df = pd.DataFrame(diag_rows)

print("NICFI availability diagnostic:")
display(nicfi_diag_df)


# ------------------------------------------------------------
# 5. Earth Engine extraction helpers
# ------------------------------------------------------------

def make_points_fc(df):
    features = []

    for _, row in df.iterrows():
        props = {
            "sample_id": str(row["sample_id"]),
            "unit_id": str(row["unit_id"]),
            "land_cover": str(row["land_cover"]),
            "observation_year": int(row["observation_year"]),
            "observation_month": int(row["observation_month"]),
            "nicfi_semester": str(row["nicfi_semester"])
        }

        if "zone_id" in row.index:
            props["zone_id"] = str(row["zone_id"])

        geom = ee.Geometry.Point([
            float(row["longitude"]),
            float(row["latitude"])
        ])

        features.append(ee.Feature(geom, props))

    return ee.FeatureCollection(features)


def extract_chunk(df_chunk, year, semester, patch_size):
    fc = make_points_fc(df_chunk)

    region = fc.geometry().bounds().buffer(1000)

    arr_img = get_patch_array_image(
        year=year,
        semester=semester,
        patch_size=patch_size,
        region=region
    )

    sampled = arr_img.sampleRegions(
        collection=fc,
        properties=[
            "sample_id",
            "unit_id",
            "land_cover",
            "observation_year",
            "observation_month",
            "nicfi_semester"
        ],
        scale=PLANET_SCALE,
        geometries=False,
        tileScale=4
    )

    info = sampled.getInfo()

    X_list = []
    sample_ids = []
    unit_ids = []
    labels = []
    years = []
    months = []
    semesters = []

    for f in info.get("features", []):
        props = f.get("properties", {})

        patch_channels = []
        ok = True

        for band in NICFI_FEATURES:
            if band not in props:
                ok = False
                break

            arr = np.array(props[band], dtype=np.float32)

            if arr.shape != (patch_size, patch_size):
                ok = False
                break

            patch_channels.append(arr)

        if not ok:
            continue

        X_list.append(np.stack(patch_channels, axis=-1))
        sample_ids.append(str(props["sample_id"]))
        unit_ids.append(str(props["unit_id"]))
        labels.append(str(props["land_cover"]))
        years.append(int(props["observation_year"]))
        months.append(int(props["observation_month"]))
        semesters.append(str(props["nicfi_semester"]))

    if len(X_list) == 0:
        return None

    return {
        "X": np.stack(X_list).astype(np.float32),
        "sample_ids": np.array(sample_ids).astype(str),
        "unit_ids": np.array(unit_ids).astype(str),
        "land_cover": np.array(labels).astype(str),
        "years": np.array(years).astype(np.int16),
        "months": np.array(months).astype(np.int8),
        "semesters": np.array(semesters).astype(str)
    }


# ------------------------------------------------------------
# 6. Build cache for one patch size
# ------------------------------------------------------------

def build_nicfi_patch_cache_for_size(patch_size):
    patch_tag = f"p{patch_size}"

    chunk_dir = os.path.join(CACHE_DIR, f"chunks_{patch_tag}")
    Path(chunk_dir).mkdir(parents=True, exist_ok=True)

    final_npz = os.path.join(
        CACHE_DIR,
        f"Kasai_reference_nicfi_biannual_relative_texture_patch_{patch_tag}_observation_year.npz"
    )

    final_meta_csv = os.path.join(
        CACHE_DIR,
        f"Kasai_reference_nicfi_biannual_relative_texture_patch_{patch_tag}_observation_year_metadata.csv"
    )

    if os.path.exists(final_npz) and os.path.exists(final_meta_csv) and not OVERWRITE_FINAL:
        print(f"Final cache already exists for {patch_tag}:")
        print(final_npz)
        print(final_meta_csv)
        data = np.load(final_npz, allow_pickle=True)
        print("Loaded X shape:", data["X"].shape)
        return final_npz

    print("\n" + "=" * 80)
    print(f"Building NICFI semester relative texture patch cache: {patch_tag}")
    print("=" * 80)

    group_cols = ["observation_year", "nicfi_semester"]

    for (year, semester), g in ref_df.groupby(group_cols):
        year = int(year)
        semester = str(semester)

        g = g.reset_index(drop=True)

        print(f"\nExtracting year {year} | {semester} | {patch_tag} | points: {len(g)}")

        n_chunks = math.ceil(len(g) / CHUNK_SIZE)

        for chunk_i in range(n_chunks):
            start_i = chunk_i * CHUNK_SIZE
            end_i = min((chunk_i + 1) * CHUNK_SIZE, len(g))

            chunk_path = os.path.join(
                chunk_dir,
                f"chunk_year{year}_{semester}_{patch_tag}_{start_i}_{end_i}.npz"
            )

            if os.path.exists(chunk_path) and not OVERWRITE_CHUNKS:
                print(f"  exists: chunk {chunk_i + 1}/{n_chunks} rows {start_i}:{end_i}")
                continue

            print(f"  extracting: chunk {chunk_i + 1}/{n_chunks} rows {start_i}:{end_i}")

            df_chunk = g.iloc[start_i:end_i].copy()

            try:
                result = extract_chunk(
                    df_chunk=df_chunk,
                    year=year,
                    semester=semester,
                    patch_size=patch_size
                )

                if result is None:
                    print("    WARNING: no valid patches returned for this chunk.")
                    continue

                np.savez_compressed(
                    chunk_path,
                    X=result["X"],
                    sample_ids=result["sample_ids"],
                    unit_ids=result["unit_ids"],
                    land_cover=result["land_cover"],
                    years=result["years"],
                    months=result["months"],
                    semesters=result["semesters"],
                    channel_names=np.array(NICFI_FEATURES).astype(str),
                    patch_size=np.array([patch_size]),
                    planet_scale=np.array([PLANET_SCALE]),
                    texture_radius_pixels=np.array([TEXTURE_RADIUS_PIXELS])
                )

                print("    saved:", os.path.basename(chunk_path), result["X"].shape)

            except Exception as e:
                print("    WARNING: extraction failed for this chunk.")
                print("    Error:", repr(e))
                print("    Leaving this chunk unsaved so it can be retried later.")

            time.sleep(0.5)

    # --------------------------------------------------------
    # Assemble final cache
    # --------------------------------------------------------

    print(f"\nAssembling final cache for {patch_tag}...")

    chunk_files = sorted([
        os.path.join(chunk_dir, f)
        for f in os.listdir(chunk_dir)
        if f.endswith(".npz")
    ])

    if len(chunk_files) == 0:
        raise ValueError(f"No chunk files found for {patch_tag} in {chunk_dir}")

    X_parts = []
    sample_parts = []
    unit_parts = []
    label_parts = []
    year_parts = []
    month_parts = []
    semester_parts = []

    for fp in tqdm(chunk_files, desc=f"Assembling {patch_tag}"):
        d = np.load(fp, allow_pickle=True)

        X_parts.append(d["X"])
        sample_parts.append(d["sample_ids"].astype(str))
        unit_parts.append(d["unit_ids"].astype(str))
        label_parts.append(d["land_cover"].astype(str))
        year_parts.append(d["years"].astype(np.int16))
        month_parts.append(d["months"].astype(np.int8))
        semester_parts.append(d["semesters"].astype(str))

    X = np.concatenate(X_parts, axis=0).astype(np.float32)
    sample_ids = np.concatenate(sample_parts).astype(str)
    unit_ids = np.concatenate(unit_parts).astype(str)
    land_cover = np.concatenate(label_parts).astype(str)
    years = np.concatenate(year_parts).astype(np.int16)
    months = np.concatenate(month_parts).astype(np.int8)
    semesters = np.concatenate(semester_parts).astype(str)

    extracted_df = pd.DataFrame({
        "sample_id": sample_ids,
        "unit_id": unit_ids,
        "land_cover": land_cover,
        "observation_year": years,
        "observation_month": months,
        "nicfi_semester": semesters,
        "extract_idx": np.arange(len(sample_ids))
    })

    # Remove duplicate extracted samples if any.
    extracted_df = extracted_df.drop_duplicates(subset=["sample_id"], keep="first").reset_index(drop=True)

    sample_ids = sample_ids[extracted_df["extract_idx"].to_numpy()]
    unit_ids = unit_ids[extracted_df["extract_idx"].to_numpy()]
    land_cover = land_cover[extracted_df["extract_idx"].to_numpy()]
    years = years[extracted_df["extract_idx"].to_numpy()]
    months = months[extracted_df["extract_idx"].to_numpy()]
    semesters = semesters[extracted_df["extract_idx"].to_numpy()]
    X = X[extracted_df["extract_idx"].to_numpy()]

    extracted_df["extract_idx"] = np.arange(len(extracted_df))

    order_df = ref_df[[
        "sample_id",
        "unit_id",
        "land_cover",
        "observation_year",
        "observation_month",
        "nicfi_semester",
        "latitude",
        "longitude"
    ]].copy()

    if "zone_id" in ref_df.columns:
        order_df["zone_id"] = ref_df["zone_id"].astype(str)

    order_df["order_idx"] = np.arange(len(order_df))

    joined = (
        order_df
        .merge(
            extracted_df[["sample_id", "extract_idx"]],
            on="sample_id",
            how="inner"
        )
        .sort_values("order_idx")
        .reset_index(drop=True)
    )

    missing_ids = sorted(set(order_df["sample_id"]) - set(joined["sample_id"]))

    print("Requested samples:", len(order_df))
    print("Extracted samples:", len(joined))
    print("Missing samples:", len(missing_ids))

    if len(missing_ids) > 0:
        print("First missing sample_ids:", missing_ids[:20])

    idx = joined["extract_idx"].to_numpy()

    X = X[idx]
    sample_ids = sample_ids[idx]
    unit_ids = unit_ids[idx]
    land_cover = land_cover[idx]
    years = years[idx]
    months = months[idx]
    semesters = semesters[idx]

    metadata_df = joined.drop(columns=["extract_idx"]).copy()

    n_nan = int(np.isnan(X).sum())
    n_inf = int(np.isinf(X).sum())

    print("Final X shape:", X.shape)
    print("NaN count:", n_nan)
    print("Inf count:", n_inf)

    if n_inf > 0:
        raise ValueError("Inf values found in X. This should not happen.")

    np.savez_compressed(
        final_npz,
        X=X,
        sample_ids=sample_ids,
        unit_ids=unit_ids,
        land_cover=land_cover,
        years=years,
        months=months,
        semesters=semesters,
        channel_names=np.array(NICFI_FEATURES).astype(str),
        patch_size=np.array([patch_size]),
        planet_scale=np.array([PLANET_SCALE]),
        texture_radius_pixels=np.array([TEXTURE_RADIUS_PIXELS])
    )

    metadata_df.to_csv(final_meta_csv, index=False)

    print("Saved final NPZ:")
    print(final_npz)

    print("Saved metadata CSV:")
    print(final_meta_csv)

    return final_npz


# ------------------------------------------------------------
# 7. Run cache extraction
# ------------------------------------------------------------

final_cache_paths = []

for patch_size in PATCH_SIZES:
    path = build_nicfi_patch_cache_for_size(patch_size)
    final_cache_paths.append(path)

print("\nDone. Final NICFI relative texture patch caches:")
for p in final_cache_paths:
    print(p)

Earth Engine initialized.
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches
PATCH_SIZES: [9, 15, 21]
CHUNK_SIZE: 40
PLANET_SCALE: 4.77
Reference rows before year filter: 3706
Reference rows after year >= 2015: 3706
Rows before duplicate sample_id removal: 3706
Rows after duplicate sample_id removal: 3706


,unit_id,latitude,longitude,observation_year,observation_month,zone_id,land_cover,source_index,observation_yearmonth_raw,sample_id,nicfi_semester
0,KASAI_000001,-4.336549,20.591859,2020,4,Ilebo,built_up,0000000000000000009e,2020-04,KASAI_000001_2020_04,H1
1,KASAI_000002,-4.333445,20.590306,2020,4,Ilebo,built_up,0000000000000000009f,2020-04,KASAI_000002_2020_04,H1
2,KASAI_000003,-4.339163,20.591668,2020,4,Ilebo,built_up,000000000000000000a0,2020-04,KASAI_000003_2020_04,H1
3,KASAI_000004,-4.341256,20.600636,2020,4,Ilebo,built_up,000000000000000000a1,2020-04,KASAI_000004_2020_04,H1
4,KASAI_000005,-4.341294,20.594186,2020,4,Ilebo,built_up,000000000000000000a2,2020-04,KASAI_000005_2020_04,H1


,land_cover,n
0,cropland,1039
1,savannah,741
2,water,738
3,forest,598
4,built_up,590


Number of NICFI derived features: 28
['NDVI', 'GNDVI', 'NDWI', 'BRIGHTNESS', 'RED_FRAC', 'GREEN_FRAC', 'BLUE_FRAC', 'NIR_FRAC', 'NDVI_MEAN', 'NDVI_STD', 'NDVI_RANGE', 'NDVI_CENTER_MINUS_MEAN', 'BRIGHTNESS_MEAN', 'BRIGHTNESS_STD', 'BRIGHTNESS_RANGE', 'BRIGHTNESS_CENTER_MINUS_MEAN', 'RED_FRAC_MEAN', 'RED_FRAC_STD', 'RED_FRAC_RANGE', 'RED_FRAC_CENTER_MINUS_MEAN', 'GREEN_FRAC_MEAN', 'GREEN_FRAC_STD', 'GREEN_FRAC_RANGE', 'GREEN_FRAC_CENTER_MINUS_MEAN', 'NIR_FRAC_MEAN', 'NIR_FRAC_STD', 'NIR_FRAC_RANGE', 'NIR_FRAC_CENTER_MINUS_MEAN']
NICFI availability diagnostic:


,year,semester,monthly_count,biannual_count
0,2020,H1,0,1
1,2020,H2,4,0
2,2021,H1,6,0
3,2021,H2,6,0
4,2022,H1,6,0
5,2022,H2,6,0
6,2023,H1,6,0
7,2023,H2,6,0
8,2024,H1,6,0
9,2024,H2,6,0


...

Requested samples: 3706
Extracted samples: 3706
Missing samples: 0
Final X shape: (3706, 9, 9, 28)
NaN count: 0
Inf count: 0
Saved final NPZ:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p9_observation_year.npz
Saved metadata CSV:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p9_observation_year_metadata.csv

Building NICFI semester relative texture patch cache: p15

Extracting year 2020 | H1 | p15 | points: 778
  exists: chunk 1/20 rows 0:40
  exists: chunk 2/20 rows 40:80
  exists: chunk 3/20 rows 80:120
  exists: chunk 4/20 rows 120:160
  exists: chunk 5/20 rows 160:200
  exists: chunk 6/20 rows 200:240
  exists: chunk 7/20 rows 240:280
  exists: chunk 8/20 rows 280:320
  exists: chunk 9/20 rows 320:360
  exists: chunk 10/20 rows 360:400
  exists: chunk 11/2

Assembling p15:   0%|          | 0/99 [00:00<?, ?it/s]

Requested samples: 3706
Extracted samples: 3706
Missing samples: 0
Final X shape: (3706, 15, 15, 28)
NaN count: 0
Inf count: 0
Saved final NPZ:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p15_observation_year.npz
Saved metadata CSV:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p15_observation_year_metadata.csv

Building NICFI semester relative texture patch cache: p21

Extracting year 2020 | H1 | p21 | points: 778
  extracting: chunk 1/20 rows 0:40
    saved: chunk_year2020_H1_p21_0_40.npz (40, 21, 21, 28)
  extracting: chunk 2/20 rows 40:80
    saved: chunk_year2020_H1_p21_40_80.npz (40, 21, 21, 28)
  extracting: chunk 3/20 rows 80:120
    saved: chunk_year2020_H1_p21_80_120.npz (40, 21, 21, 28)
  extracting: chunk 4/20 rows 120:160
    saved: chunk_year2020

Assembling p21:   0%|          | 0/99 [00:00<?, ?it/s]

Requested samples: 3706
Extracted samples: 3706
Missing samples: 0
Final X shape: (3706, 21, 21, 28)
NaN count: 0
Inf count: 0
Saved final NPZ:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p21_observation_year.npz
Saved metadata CSV:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p21_observation_year_metadata.csv

Done. Final NICFI relative texture patch caches:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p9_observation_year.npz
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Cache/nicfi_biannual_relative_texture_patches/Kasai_reference_nicfi_biannual_relative_texture_patch_p15_observation_year.npz
/content/drive/MyDrive/Colab Notebooks